<a href="https://colab.research.google.com/github/JariusF10-8/bt4222/blob/main/notebooks/features/Unified_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BT4222 — Unified Feature Engineering

One executable pipeline consolidating all feature engineering code from all group members. It preserves their feature families, resolves duplicate definitions, and documents the decisions below. Source code is pinned to the repository snapshot in the provenance cell; original notebooks are not modified.

The notebook builds **features, not trained recommenders**. It starts from the group's canonical preprocessed CSVs. If that handoff is absent, it can call the existing preprocessing function from a repository ZIP/checkout. All feature-building functions are included inline: no additional feature `.py` files need to be installed or copied.

Use a CPU runtime. Full candidate exports and historical statistics can take time and disk space; TF-IDF stays sparse and candidate exports are chunked. This deliverable was validated on canonical-preprocessing fixtures, not a full execution on your shared dataset. Outputs are cleared so the first visible run reflects your chosen input version.


## 1. Setup and run

1. Open the notebook in Colab and edit the setup cell before running it:
   - Set `STORAGE_ROOT` to the shared project folder in Drive.
   - Set `PROCESSED_VERSION` to the preprocessing version to use, such as `'v2'`.
   - Set `OUTPUT_VERSION` to a new feature-output version, such as `'unified_v2'`. Its folder must not already exist.

   `PROCESSED_DIR` and `OUTPUT_DIR` are constructed automatically from these settings.

2. Existing preprocessing is reused. If its `manifest.json` is missing and `BUILD_PROCESSED_IF_MISSING=True`, the notebook builds preprocessing outputs from the raw MIND data. A new feature build can reuse the same preprocessing version.

3. Raw data defaults to the repository’s `data/raw` folder. Set the `MIND_RAW_DIR` environment variable before running setup only if it is stored elsewhere.

4. Keep `REQUIRE_TEST=True`, `HISTORY_MODES=('train_only',)` and `ALLOW_HELDOUT_FEEDBACK=False` for the shared group build. Leave optional diagnostics disabled unless needed.

5. Run the setup cell, allow Drive access and upload a current repository ZIP when prompted. Ensure the required raw MIND folders and entity embedding files are available at the configured raw-data location.

6. Check the printed paths, then run the remaining cells from top to bottom. Keep the complete generated output folder together in Drive, including its matrices, vocabularies, dictionary and manifest.

In [ ]:
from pathlib import Path
import os
import sys
import io
import json
import csv
import math
import gc
import hashlib
import pickle
import shutil
import tempfile
import zipfile
import subprocess
import importlib
import importlib.util
import time
import warnings
from collections import Counter
from datetime import datetime, timezone

# Edit this cell once. Environment overrides take precedence over defaults.
PROCESSED_VERSION = 'v2'
OUTPUT_VERSION = 'unified_v1'  # Choose a new version for each completed build.
REQUIRE_TEST = True
BUILD_PROCESSED_IF_MISSING = True
MOUNT_GOOGLE_DRIVE = True
UPLOAD_REPOSITORY_IF_NEEDED = True
INSTALL_MISSING_DEPENDENCIES = True
CANDIDATE_CHUNK_SIZE = 100_000
MIN_SUBCATEGORY_ARTICLES = 5
USE_WEEKDAY_FEATURES = False  # Still exported for an explicit ablation.
HISTORY_MODES = ('train_only',)
ALLOW_HELDOUT_FEEDBACK = False  # Required for source Chris 'frozen'/'online' modes.
GLOBAL_COLD_START_CTR = 0.04  # Fixed initial prior; later priors use strictly past rows.
RUN_VALIDATION_DIAGNOSTICS = False
RUN_PROMINENCE_GRID = False  # Optional, validation only; never silently changes exports.
ENTITY_DIMENSION = 100
PROMINENCE_ABSTRACT_WEIGHT = 0.1  # Explicit source-Chris choice; not retuned here.
PROMINENCE_DECAY = 1.0
ENTITY_TYPES = {'person': 'P', 'organisation': 'O', 'place': 'G'}
TFIDF_PARAMS = dict(lowercase=True, stop_words='english', token_pattern=r'(?u)\b\w\w+\b',
                    ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=50_000,
                    sublinear_tf=True, norm='l2')

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    drive.mount('/content/drive')


def locate_repository():
    configured = os.environ.get('BT4222_PROJECT_ROOT')
    options = ([Path(configured).expanduser()] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    if IN_COLAB:
        options += [Path('/content/bt4222'), Path('/content/bt4222-main'),
                    Path('/content/bt4222-preprocessing')]
    return next((p.resolve() for p in options if (p / 'src/preprocessing.py').is_file()), None)


PROJECT_ROOT = locate_repository()
if PROJECT_ROOT is None and IN_COLAB and UPLOAD_REPOSITORY_IF_NEEDED:
    print('Upload a current repository ZIP from GitHub (Code -> Download ZIP on main).')
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith('.zip')]
    if len(names) != 1:
        raise ValueError('Select exactly one repository ZIP.')
    extracted = Path(tempfile.mkdtemp(prefix='bt4222_unified_', dir='/content'))
    with zipfile.ZipFile(io.BytesIO(uploaded[names[0]])) as archive:
        # Validate archive paths before extracting an uploaded ZIP.
        for member in archive.infolist():
            target = (extracted / member.filename).resolve()
            if target != extracted and extracted not in target.parents:
                raise ValueError('ZIP member points outside the extraction folder.')
        archive.extractall(extracted)
    roots = sorted({p.parent.parent for p in extracted.rglob('src/preprocessing.py')})
    if len(roots) != 1:
        raise ValueError(f'Expected one repository root; found {len(roots)}.')
    PROJECT_ROOT = roots[0]
    del uploaded
if PROJECT_ROOT is not None:
    sys.path.insert(0, str(PROJECT_ROOT))
    os.environ['BT4222_PROJECT_ROOT'] = str(PROJECT_ROOT)
    importlib.invalidate_caches()

# Shared storage location; PROJECT_ROOT still points to the code repository.
STORAGE_ROOT = Path(
    '/content/drive/MyDrive/BT4222 Group Project'
)

PROCESSED_DIR = (
    STORAGE_ROOT / 'data/processed' / PROCESSED_VERSION
)

OUTPUT_DIR = (
    STORAGE_ROOT / 'outputs/features' / OUTPUT_VERSION
)

# Keep raw embedding files in the uploaded repository.
raw_default = (
    PROJECT_ROOT / 'data/raw'
    if PROJECT_ROOT is not None
    else Path.cwd() / 'data/raw'
)
RAW_DIR = Path(
    os.environ.get('MIND_RAW_DIR', str(raw_default))
).expanduser().resolve()

if OUTPUT_DIR == PROCESSED_DIR or PROCESSED_DIR in OUTPUT_DIR.parents or OUTPUT_DIR in PROCESSED_DIR.parents:
    raise ValueError('Use an output folder separate from the preprocessing directory tree.')
if OUTPUT_DIR == RAW_DIR or RAW_DIR in OUTPUT_DIR.parents or OUTPUT_DIR in RAW_DIR.parents:
    raise ValueError('Use an output folder separate from the raw-data directory tree.')
if OUTPUT_DIR.exists():
    raise FileExistsError(
        f'{OUTPUT_DIR} already exists. '
        'Choose a new OUTPUT_VERSION for a fresh feature build; '
        'completed builds are preserved.'
    )
if not set(HISTORY_MODES).issubset({'train_only', 'frozen', 'online'}) or not HISTORY_MODES:
    raise ValueError('Select train_only, frozen and/or online history modes.')
if len(HISTORY_MODES) != len(set(HISTORY_MODES)):
    raise ValueError('History modes must be unique.')
if any(m != 'train_only' for m in HISTORY_MODES) and not ALLOW_HELDOUT_FEEDBACK:
    raise ValueError('frozen/online consume held-out feedback. Set ALLOW_HELDOUT_FEEDBACK=True only under an agreed evaluation protocol.')

requirements = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.2,<4',
                'scipy': 'scipy>=1.11,<2', 'sklearn': 'scikit-learn>=1.4,<2'}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING_DEPENDENCIES:
        raise ImportError(f'Install these dependencies before running: {missing}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
try:
    from IPython.display import display
except ImportError:
    display = print

print('Repository:', PROJECT_ROOT)
print('Preprocessed data:', PROCESSED_DIR)
print('Raw embedding files:', RAW_DIR)
print('Unified output:', OUTPUT_DIR)
print('History modes:', HISTORY_MODES)


Mounted at /content/drive
Upload a current repository ZIP from GitHub (Code -> Download ZIP on main).


Saving bt4222-main 30 October 2026.zip to bt4222-main 30 October 2026.zip
Repository: /content/bt4222_unified_fb9_95q5/bt4222-main
Preprocessed data: /content/drive/MyDrive/BT4222 Group Project/unified_feature_engineering/data/processed/v2
Raw embedding files: /content/bt4222_unified_fb9_95q5/bt4222-main/data/raw
Unified output: /content/drive/MyDrive/BT4222 Group Project/outputs/features/unified_v1
History modes: ('train_only',)


## 2. One canonical input and common integrity checks

Read `news.csv`, `events.csv` and the **complete** `candidates.csv` for each manifest partition. Retain the supplied ordered history and original candidate positions. Candidate identity is `(event_key, candidate_position, news_id)`; `event_key` includes the source file, so equal source-local impression IDs in train/dev cannot collide.

Text is preserved, including literal `NA`. JSON entity lists are validated and missingness flags are restored. Rows with missing metadata remain present. Input/output directories are separate. The output is staged and published only after matrix/CSV alignment and coverage checks pass.


In [ ]:
def require(condition, message):
    if not condition:
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def parse_annotations(value, context):
    parsed = json.loads(value) if isinstance(value, str) else value
    require(isinstance(parsed, list) and all(isinstance(e, dict) for e in parsed),
            f'{context}: expected a JSON list of entity objects')
    return parsed


def checked_boolean(series, column):
    require(series.isin(['True', 'False']).all(), f'Invalid canonical boolean in {column}')
    return series.eq('True').astype(bool)


def read_news(partition):
    news = pd.read_csv(PROCESSED_DIR / partition / 'news.csv', dtype='string', keep_default_na=False)
    require(not news.empty and news['news_id'].is_unique, f'{partition}: empty/duplicate news IDs')
    for c in news:
        if c.endswith('_missing'):
            news[c] = checked_boolean(news[c], c)
    for section in ('title', 'abstract'):
        c = section + '_entities_parsed'
        news[c] = [parse_annotations(v, f'{nid}:{c}') for nid, v in zip(news['news_id'], news[c])]
    return news.reset_index(drop=True)


def read_events(partition):
    events = pd.read_csv(PROCESSED_DIR / partition / 'events.csv',
                         usecols=['event_key', 'user_id', 'timestamp', 'history_ids', 'history_length'],
                         dtype='string', keep_default_na=False)
    require(not events.empty and events['event_key'].is_unique, f'{partition}: empty/duplicate event keys')
    events['timestamp'] = pd.to_datetime(events['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='raise')
    require(events['timestamp'].notna().all() and events['timestamp'].dt.tz is None,
            'Missing/timezone-aware timestamps: use the canonical preprocessing handoff')
    events['history_ids'] = events['history_ids'].map(json.loads)
    require(events['history_ids'].map(lambda x: isinstance(x, list) and all(isinstance(i, str) for i in x)).all(),
            f'{partition}: malformed history IDs')
    events['history_length'] = pd.to_numeric(events['history_length'], errors='raise').astype('int32')
    require(events['history_length'].eq(events['history_ids'].str.len()).all(), 'History length mismatch')
    return events.reset_index(drop=True)


KEYS = ['event_key', 'candidate_position', 'news_id']


def candidate_chunks(partition, labels=False, chunk_size=None):
    columns = KEYS + (['label'] if labels else [])
    return pd.read_csv(PROCESSED_DIR / partition / 'candidates.csv', usecols=columns,
                       dtype={'event_key': 'string', 'news_id': 'string',
                              'candidate_position': 'int32', 'label': 'int8'},
                       keep_default_na=False, chunksize=chunk_size or CANDIDATE_CHUNK_SIZE)


def read_candidates(partition, labels=False):
    columns = KEYS + (['label'] if labels else [])
    return pd.read_csv(PROCESSED_DIR / partition / 'candidates.csv', usecols=columns,
                       dtype={'event_key': 'category', 'news_id': 'category',
                              'candidate_position': 'int32', 'label': 'int8'}, keep_default_na=False)


def full_impression_counts(partition, events):
    counts = Counter()
    for batch in candidate_chunks(partition):
        require(not batch.duplicated(['event_key', 'candidate_position']).any(), 'Duplicate candidate positions')
        # Check original positions across chunk boundaries, including repeated news IDs.
        expected = batch.groupby('event_key', sort=False).cumcount().to_numpy()
        expected += batch['event_key'].map(counts).fillna(0).to_numpy(dtype='int64')
        require(np.array_equal(expected, batch['candidate_position'].to_numpy()),
                f'{partition}: candidate positions are not complete and in canonical order')
        counts.update(batch['event_key'].value_counts().to_dict())
    require(set(counts) == set(events['event_key']), f'{partition}: event/candidate coverage mismatch')
    return events['event_key'].map(counts).astype('int32')


def merge_unique(left, right, on):
    # Pandas can coerce string IDs to object when a grouped table supplies the key.
    # Standardise both sides without changing candidate identity or ordering.
    key_types = {c: 'string' for c in on if c in ('news_id', 'event_key', 'user_id')}
    left = left.astype(key_types, copy=False)
    right = right.astype(key_types, copy=False)
    common = (set(left) & set(right)) - set(on)
    require(not common, f'Unresolved overlapping columns: {sorted(common)}')
    require(not right.duplicated(on).any(), f'Lookup is not unique on {on}')
    out = left.merge(right, on=on, how='left', sort=False, validate='many_to_one')
    require(len(out) == len(left), 'A join changed the candidate count')
    return out


def normalise_rows(matrix):
    matrix = np.asarray(matrix, dtype='float32')
    norm = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.divide(matrix, norm, out=np.zeros_like(matrix), where=norm > 0)


manifest_path = PROCESSED_DIR / 'manifest.json'
if not manifest_path.is_file():
    if not BUILD_PROCESSED_IF_MISSING or PROJECT_ROOT is None:
        raise FileNotFoundError(f'{manifest_path} missing: configure an existing version or provide a repository to build one.')
    from src.preprocessing import build_dataset
    display(build_dataset(RAW_DIR, PROCESSED_DIR, require_test=REQUIRE_TEST, overwrite=False))
source_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
PARTITIONS = source_manifest.get('partitions', [])
expected = ['train', 'validation', 'test'] if REQUIRE_TEST else ['train', 'validation']
require(PARTITIONS == expected or (not REQUIRE_TEST and PARTITIONS == ['train', 'validation', 'test']),
        f'Found {PARTITIONS}; expected {expected}. Existing incomplete versions are NOT rebuilt silently: choose a new processed version.')
if REQUIRE_TEST:
    require(source_manifest.get('complete') is True, 'Manifest marks the dataset incomplete')
previous_end = None
seen_keys = set()
for partition in PARTITIONS:
    for name in ('news.csv', 'events.csv', 'candidates.csv'):
        require((PROCESSED_DIR / partition / name).is_file(), f'Missing {partition}/{name}')
    frame = read_events(partition)
    require(not (seen_keys & set(frame['event_key'])), 'Event keys overlap partitions')
    seen_keys.update(frame['event_key'])
    require(previous_end is None or previous_end < frame['timestamp'].min(), 'Chronological partitions overlap')
    previous_end = frame['timestamp'].max()
del frame, seen_keys

OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
STAGING = Path(tempfile.mkdtemp(prefix=f'.{OUTPUT_DIR.name}-building-', dir=OUTPUT_DIR.parent))
for partition in PARTITIONS:
    (STAGING / partition).mkdir()
print('Input partitions:', ', '.join(PARTITIONS))
print('Building into a staging folder; the completed version is published after checks pass.')
STARTED = time.time()

# Every exported feature is registered exactly once, with its level and source.
FEATURE_REGISTRY = {}
FILE_INVENTORY = []
SOURCE_HASHES = {}


def register_frame(frame, level, owner, excluded=(), disabled=(), descriptions=None):
    for c in frame:
        if c in excluded:
            continue
        require(c not in FEATURE_REGISTRY, f'Duplicate feature definition: {c}')
        numeric = pd.api.types.is_numeric_dtype(frame[c]) or pd.api.types.is_bool_dtype(frame[c])
        FEATURE_REGISTRY[c] = dict(feature=c, level=level, source_notebook=owner,
            dtype=str(frame[c].dtype), numeric=bool(numeric),
            default_model_input=bool(numeric and c not in disabled and c not in ('category_id', 'subcategory_id')),
            description=(descriptions or {}).get(c, c.replace('_', ' ')))


def record_file(path, level, rows=None):
    FILE_INVENTORY.append(dict(file=str(Path(path).relative_to(STAGING)), level=level,
                               rows=None if rows is None else int(rows), bytes=Path(path).stat().st_size))


def write_frame(frame, path, level, **kwargs):
    frame.to_csv(path, index=False, encoding='utf-8', **kwargs)
    record_file(path, level, len(frame))


Preparing train: 95,071 events
Preparing validation: 61,894 events
Preparing test: 73,152 events


,partition,earliest_time,latest_time,events,users,candidate_exposures,clicked_candidates,nonclicked_candidates,events_with_empty_history,referenced_articles,articles_missing_metadata,candidate_exposures_missing_metadata
0,train,2019-11-09 00:00:19,2019-11-12 23:59:25,95071,40148,3389342,141558,3247784,1735,43370,0,0
1,validation,2019-11-13 00:01:00,2019-11-14 23:59:13,61894,31664,2454102,94786,2359316,1503,38421,0,0
2,test,2019-11-15 00:00:01,2019-11-15 23:58:03,73152,50000,2740998,111383,2629615,2214,42416,0,0


Input partitions: train, validation, test
Building into a staging folder; the completed version is published after checks pass.


## 3. Overlap and conflict decisions

| Overlap / conflict | Sources | Unified decision |
|---|---|---|
| Setup and preprocessing | All five | One path/configuration cell and one canonical preprocessed handoff; no separate raw resplitting. |
| Text/entity counts | ZP + SY + preprocessing | Verify existing title_length, abstract_length, num_title_entities and num_abstract_entities, then export each once. |
| Category mapping | ZP + Chris | One sorted training-only category vocabulary with 0 unknown. Chris parent-aware subcategories retained; every known parent receives a reserved rare bucket. |
| History counts | Johannes + ZP | history_article_count is an alias of history_length. Keep history_known_category_count separately. num_candidates always comes from complete original slates. |
| Ordinary entity vectors | Ethan + Johannes + Chris | One distinct-ID uniform mean, stored as uniform. Confidence and prominence variants remain separate ablations. |
| Entity type vectors | SY | Keep per-type unit article vectors and exact-ID overlap. They are not duplicates of the unnormalised global uniform vector. |
| Metadata/embedding sources | Ethan vs the other four | Use train metadata/vectors for train+validation and dev metadata/vectors for test. Do not concatenate and keep-first across sources. |
| Candidate and event keys | All five | Use source-safe event_key and original candidate_position; rename Ethan val to validation. Labels remain in preprocessing candidates, not predictor exports. |
| Historical feedback policy | Chris vs held-out test protocol | train_only is the default: held-out labels are not read. Preserve Chris frozen and online modes as explicit opt-ins; frozen test includes validation feedback and online includes earlier test feedback. |
| Global CTR prior | Chris | Original helper uses a full-training click-rate constant. Default unified prior uses strictly past permitted feedback, preventing a current/future training label from entering its own smoothing prior. Fixed cold-start prior is 0.04. |
| Time features | ZP | Keep minutes/seconds, source-naive times, daily cycle and category/interest interactions. Export weekdays/weekend but exclude them from default model inputs. |
| Missingness and zero vectors | SY + Ethan + Johannes + Chris | Zero scores keep raw-entity and embedding-missing flags. Fix empty-entity/empty-history edge cases without dropping rows. |
| Output tables | All five | One article table, one event table, one candidate matching table, and mode-specific history tables per partition. Single matrix index and vocabulary dictionaries; no duplicated per-member output trees. |
| Diagnostics and selection | SY + Chris + Johannes | Optional train/validation diagnostics and full prominence grid retained. Remove static old results and hard-coded standard-dataset row counts. Fixed export parameters are explicit; grid recommendations do not silently change them. |
| Negative sampling | All five + src/sampling.py | Compute on complete original data. Sample only for model training; many-to-one feature joins retain repeated negatives and sampling metadata. |
| Memory and publishing | Chris + SY + ZP | Chunk candidate exports, keep TF-IDF sparse, load each embedding source once, process partitions sequentially, and publish only after integrity checks. |

| Original name | Unified name | Meaning |
|---|---|---|
| `article_entity_vector / ordinary / uniform` | `article_entity_vectors.npz::uniform` | One shared ordinary distinct-ID mean |
| `weighted (Johannes)` | `article_entity_vectors.npz::confidence` | Confidence-weighted mean, not prominence weighting |
| `entity_similarity (Ethan)` | `entity_similarity_uniform` | Candidate matching table |
| `val` | `validation` | Canonical partition name |
| `history_article_count (ZP)` | `history_length` | Event table, repeats retained |
| `category_mapping.csv / category_vocab.csv` | `category_vocab.csv` | One training-fitted vocabulary |
| `candidate_person_missing` | `person_entities_missing` | Article-level raw-annotation presence flag; join on news_id |
| `candidate_organisation_missing` | `organisation_entities_missing` | Article-level raw-annotation presence flag |
| `candidate_place_missing` | `place_entities_missing` | Article-level raw-annotation presence flag |
| `history_<type>_missing` | `history_<type>_missing` | Stored once per event instead of repeated candidate rows |
| `tfidf_news_ids.csv / news_entity_vector_index.csv` | `news_index.csv` | Shared row order for all article matrices |
| `news_entity_vectors_<selected scheme>.npy` | `article_entity_vectors.npz::prominence` | Parameters are explicit in config/manifest |
| `entity_<0..99> CSV columns` | `article_entity_vectors.npz::uniform` | Preserve float32 vector precision; do not flatten into repeated CSV columns |


In [ ]:
SOURCE_COMMIT = '7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f'
SOURCE_NOTEBOOKS = [{'path': 'notebooks/features/feature_engine_sy.ipynb', 'blob_sha': '97f17f4ffec1067acd61eb1ca51ae70b01564062', 'url': 'https://github.com/JariusF10-8/bt4222/blob/7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f/notebooks/features/feature_engine_sy.ipynb'}, {'path': 'notebooks/features/feature_engineering_chris.ipynb', 'blob_sha': '71cd5070ed23c14233432cae35cf4001bf41988f', 'url': 'https://github.com/JariusF10-8/bt4222/blob/7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f/notebooks/features/feature_engineering_chris.ipynb'}, {'path': 'notebooks/features/feature_engineering_ethan.ipynb', 'blob_sha': '492ed72204979dc19121260e5412b9d42ca0c73d', 'url': 'https://github.com/JariusF10-8/bt4222/blob/7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f/notebooks/features/feature_engineering_ethan.ipynb'}, {'path': 'notebooks/features/feature_engineering_johannes.ipynb', 'blob_sha': 'cfee6815bc1c85ea020760f6b3a1775f2a91a3b7', 'url': 'https://github.com/JariusF10-8/bt4222/blob/7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f/notebooks/features/feature_engineering_johannes.ipynb'}, {'path': 'notebooks/features/feature_engineering_zp.ipynb', 'blob_sha': '7244dd45e64ef2270bb3b1e3ef1464b54717ab69', 'url': 'https://github.com/JariusF10-8/bt4222/blob/7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f/notebooks/features/feature_engineering_zp.ipynb'}]
SOURCE_HELPERS = [{'path': 'src/preprocessing.py', 'blob_sha': '503c4d0fd20256a983a36cdcbaa1fa0e91055d3a'}, {'path': 'src/data_loading.py', 'blob_sha': '1d3c7e110095dc81cbf19ebd0bffa59bfaa9fff9'}, {'path': 'src/sampling.py', 'blob_sha': '3ba9285b1773d881058b1ec18fd116cbe1df6db3'}, {'path': 'src/category_encoding.py', 'blob_sha': 'bdc83866e7db5c948c818a3ab03b7c5389fc69e5'}, {'path': 'src/history_features.py', 'blob_sha': '47a4d13f2472cc04e36960be01ec0186a4d8eb5c'}, {'path': 'src/entity_prominence.py', 'blob_sha': '667c6d98d36a2f3821e77497688d67bf9b6e910c'}, {'path': 'src/feature_diagnostics.py', 'blob_sha': '17e6690c53503ec2c837e9a1aa869f17150a3fb3'}]
MERGE_DECISIONS = [{'topic': 'Setup and preprocessing', 'sources': 'All five', 'decision': 'One path/configuration cell and one canonical preprocessed handoff; no separate raw resplitting.'}, {'topic': 'Text/entity counts', 'sources': 'ZP + SY + preprocessing', 'decision': 'Verify existing title_length, abstract_length, num_title_entities and num_abstract_entities, then export each once.'}, {'topic': 'Category mapping', 'sources': 'ZP + Chris', 'decision': 'One sorted training-only category vocabulary with 0 unknown. Chris parent-aware subcategories retained; every known parent receives a reserved rare bucket.'}, {'topic': 'History counts', 'sources': 'Johannes + ZP', 'decision': 'history_article_count is an alias of history_length. Keep history_known_category_count separately. num_candidates always comes from complete original slates.'}, {'topic': 'Ordinary entity vectors', 'sources': 'Ethan + Johannes + Chris', 'decision': 'One distinct-ID uniform mean, stored as uniform. Confidence and prominence variants remain separate ablations.'}, {'topic': 'Entity type vectors', 'sources': 'SY', 'decision': 'Keep per-type unit article vectors and exact-ID overlap. They are not duplicates of the unnormalised global uniform vector.'}, {'topic': 'Metadata/embedding sources', 'sources': 'Ethan vs the other four', 'decision': 'Use train metadata/vectors for train+validation and dev metadata/vectors for test. Do not concatenate and keep-first across sources.'}, {'topic': 'Candidate and event keys', 'sources': 'All five', 'decision': 'Use source-safe event_key and original candidate_position; rename Ethan val to validation. Labels remain in preprocessing candidates, not predictor exports.'}, {'topic': 'Historical feedback policy', 'sources': 'Chris vs held-out test protocol', 'decision': 'train_only is the default: held-out labels are not read. Preserve Chris frozen and online modes as explicit opt-ins; frozen test includes validation feedback and online includes earlier test feedback.'}, {'topic': 'Global CTR prior', 'sources': 'Chris', 'decision': 'Original helper uses a full-training click-rate constant. Default unified prior uses strictly past permitted feedback, preventing a current/future training label from entering its own smoothing prior. Fixed cold-start prior is 0.04.'}, {'topic': 'Time features', 'sources': 'ZP', 'decision': 'Keep minutes/seconds, source-naive times, daily cycle and category/interest interactions. Export weekdays/weekend but exclude them from default model inputs.'}, {'topic': 'Missingness and zero vectors', 'sources': 'SY + Ethan + Johannes + Chris', 'decision': 'Zero scores keep raw-entity and embedding-missing flags. Fix empty-entity/empty-history edge cases without dropping rows.'}, {'topic': 'Output tables', 'sources': 'All five', 'decision': 'One article table, one event table, one candidate matching table, and mode-specific history tables per partition. Single matrix index and vocabulary dictionaries; no duplicated per-member output trees.'}, {'topic': 'Diagnostics and selection', 'sources': 'SY + Chris + Johannes', 'decision': 'Optional train/validation diagnostics and full prominence grid retained. Remove static old results and hard-coded standard-dataset row counts. Fixed export parameters are explicit; grid recommendations do not silently change them.'}, {'topic': 'Negative sampling', 'sources': 'All five + src/sampling.py', 'decision': 'Compute on complete original data. Sample only for model training; many-to-one feature joins retain repeated negatives and sampling metadata.'}, {'topic': 'Memory and publishing', 'sources': 'Chris + SY + ZP', 'decision': 'Chunk candidate exports, keep TF-IDF sparse, load each embedding source once, process partitions sequentially, and publish only after integrity checks.'}]
ALIAS_RECORDS = [{'source_name': 'article_entity_vector / ordinary / uniform', 'unified_name': 'article_entity_vectors.npz::uniform', 'note': 'One shared ordinary distinct-ID mean'}, {'source_name': 'weighted (Johannes)', 'unified_name': 'article_entity_vectors.npz::confidence', 'note': 'Confidence-weighted mean, not prominence weighting'}, {'source_name': 'entity_similarity (Ethan)', 'unified_name': 'entity_similarity_uniform', 'note': 'Candidate matching table'}, {'source_name': 'val', 'unified_name': 'validation', 'note': 'Canonical partition name'}, {'source_name': 'history_article_count (ZP)', 'unified_name': 'history_length', 'note': 'Event table, repeats retained'}, {'source_name': 'category_mapping.csv / category_vocab.csv', 'unified_name': 'category_vocab.csv', 'note': 'One training-fitted vocabulary'}, {'source_name': 'candidate_person_missing', 'unified_name': 'person_entities_missing', 'note': 'Article-level raw-annotation presence flag; join on news_id'}, {'source_name': 'candidate_organisation_missing', 'unified_name': 'organisation_entities_missing', 'note': 'Article-level raw-annotation presence flag'}, {'source_name': 'candidate_place_missing', 'unified_name': 'place_entities_missing', 'note': 'Article-level raw-annotation presence flag'}, {'source_name': 'history_<type>_missing', 'unified_name': 'history_<type>_missing', 'note': 'Stored once per event instead of repeated candidate rows'}, {'source_name': 'tfidf_news_ids.csv / news_entity_vector_index.csv', 'unified_name': 'news_index.csv', 'note': 'Shared row order for all article matrices'}, {'source_name': 'news_entity_vectors_<selected scheme>.npy', 'unified_name': 'article_entity_vectors.npz::prominence', 'note': 'Parameters are explicit in config/manifest'}, {'source_name': 'entity_<0..99> CSV columns', 'unified_name': 'article_entity_vectors.npz::uniform', 'note': 'Preserve float32 vector precision; do not flatten into repeated CSV columns'}]
MERGE_NOTES = '# Consolidation notes\n\nSnapshot: 7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f\n\n| Overlap / conflict | Sources | Unified decision |\n|---|---|---|\n| Setup and preprocessing | All five | One path/configuration cell and one canonical preprocessed handoff; no separate raw resplitting. |\n| Text/entity counts | ZP + SY + preprocessing | Verify existing title_length, abstract_length, num_title_entities and num_abstract_entities, then export each once. |\n| Category mapping | ZP + Chris | One sorted training-only category vocabulary with 0 unknown. Chris parent-aware subcategories retained; every known parent receives a reserved rare bucket. |\n| History counts | Johannes + ZP | history_article_count is an alias of history_length. Keep history_known_category_count separately. num_candidates always comes from complete original slates. |\n| Ordinary entity vectors | Ethan + Johannes + Chris | One distinct-ID uniform mean, stored as uniform. Confidence and prominence variants remain separate ablations. |\n| Entity type vectors | SY | Keep per-type unit article vectors and exact-ID overlap. They are not duplicates of the unnormalised global uniform vector. |\n| Metadata/embedding sources | Ethan vs the other four | Use train metadata/vectors for train+validation and dev metadata/vectors for test. Do not concatenate and keep-first across sources. |\n| Candidate and event keys | All five | Use source-safe event_key and original candidate_position; rename Ethan val to validation. Labels remain in preprocessing candidates, not predictor exports. |\n| Historical feedback policy | Chris vs held-out test protocol | train_only is the default: held-out labels are not read. Preserve Chris frozen and online modes as explicit opt-ins; frozen test includes validation feedback and online includes earlier test feedback. |\n| Global CTR prior | Chris | Original helper uses a full-training click-rate constant. Default unified prior uses strictly past permitted feedback, preventing a current/future training label from entering its own smoothing prior. Fixed cold-start prior is 0.04. |\n| Time features | ZP | Keep minutes/seconds, source-naive times, daily cycle and category/interest interactions. Export weekdays/weekend but exclude them from default model inputs. |\n| Missingness and zero vectors | SY + Ethan + Johannes + Chris | Zero scores keep raw-entity and embedding-missing flags. Fix empty-entity/empty-history edge cases without dropping rows. |\n| Output tables | All five | One article table, one event table, one candidate matching table, and mode-specific history tables per partition. Single matrix index and vocabulary dictionaries; no duplicated per-member output trees. |\n| Diagnostics and selection | SY + Chris + Johannes | Optional train/validation diagnostics and full prominence grid retained. Remove static old results and hard-coded standard-dataset row counts. Fixed export parameters are explicit; grid recommendations do not silently change them. |\n| Negative sampling | All five + src/sampling.py | Compute on complete original data. Sample only for model training; many-to-one feature joins retain repeated negatives and sampling metadata. |\n| Memory and publishing | Chris + SY + ZP | Chunk candidate exports, keep TF-IDF sparse, load each embedding source once, process partitions sequentially, and publish only after integrity checks. |\n\n## Name aliases\n\n| Original name | Unified name | Meaning |\n|---|---|---|\n| `article_entity_vector / ordinary / uniform` | `article_entity_vectors.npz::uniform` | One shared ordinary distinct-ID mean |\n| `weighted (Johannes)` | `article_entity_vectors.npz::confidence` | Confidence-weighted mean, not prominence weighting |\n| `entity_similarity (Ethan)` | `entity_similarity_uniform` | Candidate matching table |\n| `val` | `validation` | Canonical partition name |\n| `history_article_count (ZP)` | `history_length` | Event table, repeats retained |\n| `category_mapping.csv / category_vocab.csv` | `category_vocab.csv` | One training-fitted vocabulary |\n| `candidate_person_missing` | `person_entities_missing` | Article-level raw-annotation presence flag; join on news_id |\n| `candidate_organisation_missing` | `organisation_entities_missing` | Article-level raw-annotation presence flag |\n| `candidate_place_missing` | `place_entities_missing` | Article-level raw-annotation presence flag |\n| `history_<type>_missing` | `history_<type>_missing` | Stored once per event instead of repeated candidate rows |\n| `tfidf_news_ids.csv / news_entity_vector_index.csv` | `news_index.csv` | Shared row order for all article matrices |\n| `news_entity_vectors_<selected scheme>.npy` | `article_entity_vectors.npz::prominence` | Parameters are explicit in config/manifest |\n| `entity_<0..99> CSV columns` | `article_entity_vectors.npz::uniform` | Preserve float32 vector precision; do not flatten into repeated CSV columns |'


## 4. Shared category and subcategory vocabulary

Fit on training articles with metadata only. Category ID 0 is unknown. Subcategory IDs encode **(parent category, subcategory)** pairs, with a rare bucket reserved for each training category. `MIN_SUBCATEGORY_ARTICLES=5` preserves the source threshold. The extra reserved buckets fix the case where a category had no rare training pair but later receives a new subcategory; some subcategory IDs therefore differ from the original notebook. Use this bundle's saved vocabulary throughout.

Raw subcategory matching remains label-free and is not merged merely because two subcategories share a rare bucket. Category one-hots are stored once in article features; time cross-products reuse the same codes.


In [ ]:
"""Category and subcategory encodings fitted on training articles only.

category_id follows the same rule as feature_engineering_zp.ipynb: sorted
training categories get 1..K and 0 is reserved for unknown. Subcategories are
encoded as (category, subcategory) pairs because 13 subcategory names appear
under more than one category. Pairs with fewer than min_articles training
articles share one rare bucket per category.
"""
import numpy as np
import pandas as pd


UNKNOWN = "__UNKNOWN__"
RARE = "__RARE__"


def _known(news):
    category = news["category"].astype("string").fillna("")
    return (~news["metadata_missing"].astype(bool)) & category.str.strip().ne("") & category.ne(UNKNOWN)


def fit_category_vocab(train_news, min_articles=5):
    require(min_articles >= 1, "min_articles must be at least 1")
    require(train_news["news_id"].is_unique, "Duplicate news_id in training news")
    known = train_news.loc[_known(train_news), ["category", "subcategory"]].astype(str)
    require(len(known) > 0, "No training articles with a known category")

    categories = sorted(known["category"].unique())
    category_vocab = pd.DataFrame({
        "category_id": np.arange(len(categories) + 1, dtype="int32"),
        "category": [UNKNOWN, *categories],
    })
    category_vocab["train_articles"] = category_vocab["category"].map(
        known["category"].value_counts()).fillna(0).astype("int64")

    pairs = known.loc[known["subcategory"].ne(UNKNOWN)].value_counts(["category", "subcategory"]).rename("train_articles").reset_index()
    pairs["bucket"] = np.where(pairs["train_articles"].ge(min_articles), pairs["subcategory"], RARE)
    buckets = (pairs.groupby(["category", "bucket"], as_index=False)
               .agg(train_articles=("train_articles", "sum"), merged_subcategories=("subcategory", "size"))
               .sort_values(["category", "bucket"], kind="stable").reset_index(drop=True))
    # Unified correction: reserve a rare bucket even if a category has no rare training pairs.
    present = set(zip(buckets["category"], buckets["bucket"]))
    missing_buckets = [{"category": c, "bucket": RARE, "train_articles": 0, "merged_subcategories": 0}
                       for c in categories if (c, RARE) not in present]
    if missing_buckets:
        buckets = pd.concat([buckets, pd.DataFrame(missing_buckets)], ignore_index=True)
        buckets = buckets.sort_values(["category", "bucket"], kind="stable").reset_index(drop=True)
    buckets.insert(0, "subcategory_id", np.arange(1, len(buckets) + 1, dtype="int32"))
    unknown_row = pd.DataFrame([{"subcategory_id": 0, "category": UNKNOWN, "bucket": UNKNOWN,
                                 "train_articles": 0, "merged_subcategories": 0}])
    subcategory_vocab = pd.concat([unknown_row, buckets], ignore_index=True)
    subcategory_vocab["subcategory_id"] = subcategory_vocab["subcategory_id"].astype("int32")
    pair_map = pairs[["category", "subcategory", "bucket", "train_articles"]].reset_index(drop=True)
    return {"category": category_vocab, "subcategory": subcategory_vocab,
            "pairs": pair_map, "min_articles": int(min_articles)}


def encode_categories(news, vocab):
    """One row per news_id with integer codes for embedding layers or tree models.

    Unseen categories map to 0. A known category with an unseen or rare
    subcategory maps to that category's rare bucket when the bucket exists,
    otherwise to 0, and subcategory_backoff records that it was not encoded
    individually.
    """
    require(news["news_id"].is_unique, "Duplicate news_id would multiply joins")
    out = news[["news_id"]].copy()
    known = _known(news)
    category = news["category"].astype(str)
    subcategory = news["subcategory"].astype(str)

    cat_ids = dict(zip(category_vocab_names(vocab), vocab["category"]["category_id"]))
    out["category_id"] = np.where(known, category.map(cat_ids).fillna(0), 0).astype("int32")

    sub_vocab = vocab["subcategory"]
    bucket_ids = {(c, b): i for c, b, i in zip(sub_vocab["category"], sub_vocab["bucket"], sub_vocab["subcategory_id"])}
    pair_bucket = {(c, s): b for c, s, b in zip(vocab["pairs"]["category"], vocab["pairs"]["subcategory"], vocab["pairs"]["bucket"])}
    codes, backoff = [], []
    for is_known, c, s in zip(known, category, subcategory):
        if not is_known or c not in cat_ids:
            codes.append(0)
            backoff.append(True)
            continue
        bucket = pair_bucket.get((c, s), RARE)
        code = bucket_ids.get((c, bucket), bucket_ids.get((c, RARE), 0))
        codes.append(code)
        backoff.append(bucket == RARE)
    out["subcategory_id"] = np.asarray(codes, dtype="int32")
    out["category_unknown"] = out["category_id"].eq(0).astype("int8")
    out["subcategory_backoff"] = np.asarray(backoff, dtype="int8")
    return out


def category_vocab_names(vocab):
    return vocab["category"]["category"].tolist()


def subcategory_key(news):
    """Raw (category, subcategory) label used for label-free history matching."""
    known = _known(news) & news["subcategory"].astype(str).ne(UNKNOWN)
    key = news["category"].astype(str) + "/" + news["subcategory"].astype(str)
    return key.where(known, "")


def candidate_history_subcategory_match(events, candidates, news):
    """Share of the impression's supplied history in the candidate's subcategory.

    Uses only the supplied prior history and article metadata, never labels.
    Matching uses raw (category, subcategory) pairs, so it needs no fitted
    vocabulary. Unknown subcategories never count as a match. Rows keep the
    candidates order.
    """
    key_of = dict(zip(news["news_id"].astype(str), subcategory_key(news)))
    hist = events[["event_key", "history_ids"]].explode("history_ids").dropna()
    hist["key"] = hist["history_ids"].astype(str).map(key_of).fillna("")
    hist = hist.loc[hist["key"].ne("")]
    known_total = hist.groupby("event_key").size().rename("history_subcategory_known")
    counts = hist.groupby(["event_key", "key"]).size().rename("candidate_history_subcategory_count").reset_index()

    out = candidates[["event_key", "candidate_position", "news_id"]].copy()
    out["key"] = out["news_id"].astype(str).map(key_of).fillna("")
    out = out.merge(counts, on=["event_key", "key"], how="left", validate="many_to_one", sort=False)
    out = out.merge(known_total, left_on="event_key", right_index=True, how="left", sort=False)
    require(len(out) == len(candidates), "Subcategory match join changed the candidate count")
    out["candidate_history_subcategory_count"] = out["candidate_history_subcategory_count"].fillna(0).astype("int32")
    total = out.pop("history_subcategory_known").fillna(0).astype("int32")
    out["history_subcategory_missing"] = total.eq(0).astype("int8")
    out["candidate_history_subcategory_share"] = np.where(
        total.gt(0), out["candidate_history_subcategory_count"] / total.clip(lower=1), 0.0).astype("float32")
    out.loc[out["key"].eq(""), ["candidate_history_subcategory_count", "candidate_history_subcategory_share"]] = 0
    return out.drop(columns="key")



## 5. Entity prominence

Keep title and abstract character positions separate. A located entity is at the title start when only punctuation/whitespace precedes it. The strict offset-0 interpretation can differ from this source definition; see `offset_audit.csv` and the documented flag meaning.

The weighting formula is section weight times `exp(-decay × normalised_position)`. Unknown positions use 1.0 (least prominent). Deduplicate each Wikidata ID using its maximum mention weight. Uniform, confidence and prominence weighting are separate variants, never accidentally combined. The explicit prominence default retains abstracts with weight 0.1 and decay 1, following the source Chris recommendation; its benefit still needs controlled model evaluation.


In [ ]:
"""Entity prominence from OccurrenceOffsets and position-weighted TransE aggregation.

Offsets are character positions in the raw title or abstract. Title and
abstract positions are kept separate and normalised by that text's length.
Earlier position is treated as a hypothesis about importance, so weighting
schemes are compared rather than assumed.
"""
from pathlib import Path

import numpy as np
import pandas as pd


SECTIONS = ("title", "abstract")
TYPE_GROUPS = {"P": "person", "O": "organisation", "G": "place"}
MENTION_COLUMNS = ["news_id", "section", "wikidata_id", "entity_type", "confidence",
                   "first_offset", "occurrences", "text_chars", "position", "at_text_start"]


def is_text_start(text, offset):
    """True when no letter or digit precedes the offset (allows a leading quote)."""
    return offset >= 0 and not any(ch.isalnum() for ch in text[:offset])


def entity_mentions(news):
    """One row per entity annotation, with its earliest position in its own section."""
    rows = []
    for news_id, *fields in zip(news["news_id"], *[news[s] for s in SECTIONS],
                                *[news[f"{s}_entities_parsed"] for s in SECTIONS]):
        texts, parsed = fields[:2], fields[2:]
        for section, text, entities in zip(SECTIONS, texts, parsed):
            chars = len(text)
            for entity in entities:
                offsets = [o for o in entity.get("OccurrenceOffsets") or [] if isinstance(o, int) and o >= 0]
                first = min(offsets) if offsets else -1
                rows.append((
                    news_id, section, str(entity.get("WikidataId") or ""), str(entity.get("Type") or ""),
                    float(entity.get("Confidence", np.nan)), first, len(offsets), chars,
                    min(first / chars, 1.0) if first >= 0 and chars > 0 else np.nan,
                    is_text_start(text, first) if first >= 0 else False,
                ))
    mentions = pd.DataFrame(rows, columns=MENTION_COLUMNS)
    mentions["first_offset"] = mentions["first_offset"].astype("int32")
    return mentions


def article_prominence_features(news, mentions):
    """Per-article prominence features; absent sections get position 1.0 plus a flag."""
    out = news[["news_id"]].copy()
    title = mentions.loc[mentions["section"].eq("title") & mentions["first_offset"].ge(0)]
    lead = title.loc[title["at_text_start"]].sort_values(["news_id", "first_offset"], kind="stable")
    lead = lead.drop_duplicates("news_id").set_index("news_id")["entity_type"]
    out["has_entity_at_title_start"] = out["news_id"].isin(lead.index).astype("int8")
    lead_type = out["news_id"].map(lead).fillna("")
    out["title_start_entity_type"] = lead_type
    out["title_start_type_group"] = np.where(
        lead_type.eq(""), "none", lead_type.map(TYPE_GROUPS).fillna("other"))
    for section in SECTIONS:
        located = mentions.loc[mentions["section"].eq(section) & mentions["position"].notna()]
        earliest = located.groupby("news_id")["position"].min()
        position = out["news_id"].map(earliest)
        out[f"{section}_entity_absent"] = position.isna().astype("int8")
        out[f"{section}_entity_first_position"] = position.fillna(1.0).astype("float32")
    return out


def mention_weights(mentions, abstract_weight=0.5, decay=1.0, use_confidence=False):
    """w = section weight x exp(-decay x normalised position) [x confidence].

    decay=0 and abstract_weight=1 reproduce uniform averaging. Mentions with
    unknown offsets are treated as the least prominent position (1.0).
    """
    section = np.where(mentions["section"].eq("title"), 1.0, abstract_weight)
    position = mentions["position"].fillna(1.0).to_numpy()
    weights = section * np.exp(-decay * position)
    if use_confidence:
        weights = weights * mentions["confidence"].fillna(1.0).to_numpy()
    return pd.Series(weights, index=mentions.index, name="weight")


def load_entity_vectors(path):
    ids, rows = [], []
    with Path(path).open(encoding="utf-8") as stream:
        for line in stream:
            parts = line.split()
            if len(parts) > 1:
                ids.append(parts[0])
                rows.append(np.asarray(parts[1:], dtype="float32"))
    require(len({len(r) for r in rows}) == 1, f"Inconsistent vector lengths in {path}")
    require(len(set(ids)) == len(ids), f"Repeated entity IDs in {path}")
    return {entity: i for i, entity in enumerate(ids)}, np.vstack(rows)


def aggregate_entity_vectors(news_ids, mentions, weights, entity_index, matrix):
    """Weighted mean of TransE vectors per article.

    An entity named in both title and abstract counts once, with its largest
    weight. Entities without a pretrained vector are skipped. Returns the
    matrix aligned to news_ids and an int8 flag for articles with no vector.
    """
    frame = mentions[["news_id", "wikidata_id"]].assign(weight=weights.to_numpy())
    frame = frame.loc[frame["wikidata_id"].isin(entity_index) & frame["weight"].gt(0)]
    frame = frame.groupby(["news_id", "wikidata_id"], as_index=False)["weight"].max()
    row_of = pd.Series(np.arange(len(news_ids)), index=pd.Index(news_ids, name="news_id"))
    frame = frame.loc[frame["news_id"].isin(row_of.index)]
    rows = row_of.loc[frame["news_id"]].to_numpy(dtype="int64")
    cols = frame["wikidata_id"].map(entity_index).to_numpy(dtype="int64")
    w = frame["weight"].to_numpy(dtype="float32")
    total = np.zeros((len(news_ids), matrix.shape[1]), dtype="float32")
    np.add.at(total, rows, w[:, None] * matrix[cols])
    mass = np.bincount(rows, weights=w, minlength=len(news_ids)).astype("float32")
    vectors = np.divide(total, mass[:, None], out=np.zeros_like(total), where=mass[:, None] > 0)
    return vectors, (mass == 0).astype("int8")



## 6. Article features, text and entity representations

- Verify and store the four basic text/entity counts once. Count annotation objects, not repeated mentions or offsets.
- Fit title+abstract TF-IDF on training metadata only: unigrams/bigrams, English stop words, `min_df=3`, `max_df=0.9`, at most 50,000 terms, sublinear TF, L2 normalisation, float32. No automatic parameter fallback if the configured corpus is too small.
- `uniform`: average distinct available entity embeddings. This replaces the duplicate Ethan/Johannes/Chris ordinary vectors.
- `confidence`: maximum linking confidence per distinct entity, as Johannes specified; keep embedded-entity count, mean/min confidence and explicit missingness.
- `prominence` and `title_only`: separate weighted variants for ablations.
- `type_person`, `type_organisation`, `type_place`: SY's L2-normalised per-type article vectors. Per-type exact-ID sets include linked entities even when an embedding is absent; additional embedding-missing flags record that distinction.

Train/validation metadata and entity vectors come from the training source; test uses dev. Shared entity vectors are compared and any differences reported. No relation embeddings are used because no entity–relation edge table is supplied by these notebooks.


In [ ]:
def derive_basic_article_features(news):
    out = news[['news_id', 'category', 'subcategory']].copy()
    for flag in ('metadata_missing', 'title_missing', 'abstract_missing',
                 'title_entities_missing', 'abstract_entities_missing'):
        out[flag] = news[flag].astype('int8')
    for section in ('title', 'abstract'):
        derived = {
            f'{section}_length': news[section].fillna('').str.split().str.len().astype('int32'),
            f'num_{section}_entities': news[f'{section}_entities_parsed'].str.len().astype('int32'),
        }
        for column, values in derived.items():
            if column in news:
                require(pd.to_numeric(news[column], errors='raise').eq(values).all(),
                        f'{column} disagrees with canonical preprocessing')
            out[column] = values
    return out


def title_abstract_text(news):
    text = (news['title'].str.strip() + ' ' + news['abstract'].str.strip()).str.strip()
    return text.mask(news['metadata_missing'], '')


train_news = read_news('train')
category_vocab = fit_category_vocab(train_news, min_articles=MIN_SUBCATEGORY_ARTICLES)
CATEGORY_COUNT = len(category_vocab['category'])
CATEGORY_ONEHOT_COLUMNS = [f'candidate_category_{i:02d}' for i in range(CATEGORY_COUNT)]
HISTORY_CATEGORY_SHARE_COLUMNS = [f'history_category_share_{i:02d}' for i in range(CATEGORY_COUNT)]
tfidf_fit_text = title_abstract_text(train_news).loc[~train_news['metadata_missing']]
vectorizer = TfidfVectorizer(**TFIDF_PARAMS, dtype=np.float32)
try:
    vectorizer.fit(tfidf_fit_text)
except ValueError as exc:
    raise ValueError('TF-IDF could not fit with the configured min_df/max_df. Inspect the training corpus or explicitly adjust TFIDF_PARAMS; no silent fallback is used.') from exc
vocabulary = vectorizer.get_feature_names_out()
FIT_ARTICLES = len(tfidf_fit_text)
write_frame(category_vocab['category'], STAGING / 'category_vocab.csv', 'vocabulary')
write_frame(category_vocab['subcategory'], STAGING / 'subcategory_vocab.csv', 'vocabulary')
write_frame(category_vocab['pairs'], STAGING / 'subcategory_pairs.csv', 'vocabulary')
write_frame(pd.DataFrame({'column': np.arange(len(vocabulary)), 'term': vocabulary}),
            STAGING / 'tfidf_vocabulary.csv', 'vocabulary')
with (STAGING / 'tfidf_vectorizer.pkl').open('wb') as stream:
    pickle.dump(vectorizer, stream)
record_file(STAGING / 'tfidf_vectorizer.pkl', 'fitted transform')
print(f'Training-only vocabulary: {CATEGORY_COUNT - 1} categories, {len(vocabulary):,} TF-IDF terms')
del train_news, tfidf_fit_text

EMBEDDING_CACHE = {}
EMBEDDING_SOURCES = {'train': 'MINDsmall_train', 'validation': 'MINDsmall_train', 'test': 'MINDsmall_dev'}
EMBEDDING_AUDIT = []
VECTOR_SCHEMES = {'uniform': (1.0, 0.0),
                  'prominence': (PROMINENCE_ABSTRACT_WEIGHT, PROMINENCE_DECAY),
                  'title_only': (0.0, 1.0)}


def embeddings_for(partition):
    source = EMBEDDING_SOURCES[partition]
    if source not in EMBEDDING_CACHE:
        path = RAW_DIR / source / 'entity_embedding.vec'
        require(path.is_file(), f'Missing {path}. Set MIND_RAW_DIR to the folder containing both raw MIND source folders.')
        lookup, matrix = load_entity_vectors(path)
        require(matrix.ndim == 2 and matrix.shape[1] == ENTITY_DIMENSION,
                f'{path}: expected {ENTITY_DIMENSION} dimensions')
        require(np.isfinite(matrix).all(), f'{path}: non-finite entity embeddings')
        EMBEDDING_CACHE[source] = lookup, matrix
        SOURCE_HASHES[str(path)] = sha256_file(path)
        if source == 'MINDsmall_dev' and 'MINDsmall_train' in EMBEDDING_CACHE:
            left, a = EMBEDDING_CACHE['MINDsmall_train']
            shared = sorted(set(left) & set(lookup))
            gap = float(np.max(np.abs(a[[left[q] for q in shared]] - matrix[[lookup[q] for q in shared]]))) if shared else None
            EMBEDDING_AUDIT.append({'shared_entities': len(shared), 'largest_absolute_difference': gap})
            if gap is not None and gap > 1e-6:
                warnings.warn('Train/dev supplied entity vectors differ on shared IDs. Partition-specific source tables are preserved; inspect this before comparisons.')
    return EMBEDDING_CACHE[source]


def derive_entity_representations(news, entity_index, matrix):
    news_ids = news['news_id'].tolist()
    mentions = entity_mentions(news)
    # Validate linked-entity confidence rather than silently treating malformed values as 1.
    conf = mentions['confidence'].to_numpy(dtype=float)
    require(np.isfinite(conf).all() and ((conf >= 0) & (conf <= 1)).all(), 'Invalid/missing entity-linking confidence')
    vectors, flags = {}, {}
    for name, (abstract_weight, decay) in VECTOR_SCHEMES.items():
        weights = mention_weights(mentions, abstract_weight=abstract_weight, decay=decay)
        vectors[name], flags[name] = aggregate_entity_vectors(news_ids, mentions, weights, entity_index, matrix)
    # Shared ordinary vector: Ethan = Johannes ordinary = Chris uniform, stored once.
    confidences = mentions.loc[mentions['wikidata_id'].isin(entity_index)]
    confidences = confidences.groupby(['news_id', 'wikidata_id'], as_index=False, sort=False)['confidence'].max()
    summary = confidences.groupby('news_id')['confidence'].agg(
        num_embedded_entities='size', entity_conf_mean='mean', entity_conf_min='min')
    summary = summary.reindex(news_ids, fill_value=0).reset_index()
    summary['num_embedded_entities'] = summary['num_embedded_entities'].astype('int32')
    summary[['entity_conf_mean', 'entity_conf_min']] = summary[['entity_conf_mean', 'entity_conf_min']].astype('float32')
    confidence_weights = pd.Series(mentions['confidence'].to_numpy(), index=mentions.index)
    vectors['confidence'], flags['confidence'] = aggregate_entity_vectors(
        news_ids, mentions, confidence_weights, entity_index, matrix)
    # Same dedup rule as Johannes: maximum annotation confidence for each distinct ID.
    summary['entity_conf_missing'] = summary['num_embedded_entities'].eq(0).astype('int8')
    for name in vectors:
        # Explicitly flag zero/cancelling vectors as unavailable for cosine matching.
        summary[f'entity_vector_missing_{name}'] = (np.linalg.norm(vectors[name], axis=1) == 0).astype('int8')
    prominence = article_prominence_features(news, mentions)
    features = merge_unique(summary, prominence, ['news_id'])
    for group in ('person', 'organisation', 'place', 'other', 'none'):
        features[f'title_start_type_{group}'] = features['title_start_type_group'].eq(group).astype('int8')

    entities = [title + abstract for title, abstract in zip(news['title_entities_parsed'], news['abstract_entities_parsed'])]
    incidence, type_vectors, type_descriptions = {}, {}, {'news_id': news_ids}
    for name, code in ENTITY_TYPES.items():
        ids = [list(dict.fromkeys(str(e['WikidataId']) for e in row
               if e.get('Type') == code and isinstance(e.get('WikidataId'), str) and e['WikidataId']))
               for row in entities]
        vocab = pd.Index(sorted({q for row in ids for q in row}))
        lengths = np.array([len(row) for row in ids], dtype='int32')
        flat = [q for row in ids for q in row]
        rr = np.repeat(np.arange(len(ids), dtype='int64'), lengths)
        cc = vocab.get_indexer(flat).astype('int64')
        inc = sparse.csr_matrix((np.ones(len(flat), dtype='float32'), (rr, cc)),
                                shape=(len(ids), len(vocab)))
        available = np.array([q in entity_index for q in vocab], dtype=bool)
        embedded = np.zeros((len(vocab), matrix.shape[1]), dtype='float32')
        if available.any():
            embedded[available] = matrix[[entity_index[q] for q in vocab[available]]]
        incidence[name] = inc
        # Preserve SY's equal-weight UNIT article vectors, distinct from global ordinary vectors.
        type_vectors[name] = normalise_rows(np.asarray(inc @ embedded, dtype='float32'))
        features[f'num_{name}_entities'] = lengths
        features[f'{name}_entities_missing'] = (lengths == 0).astype('int8')
        features[f'{name}_entities_without_embedding'] = np.asarray(inc[:, ~available].sum(axis=1)).ravel().astype('int32')
        features[f'{name}_embedding_missing'] = (np.linalg.norm(type_vectors[name], axis=1) == 0).astype('int8')
        type_descriptions[f'{name}_entity_ids'] = [json.dumps(row) for row in ids]
    return features, vectors, incidence, type_vectors, pd.DataFrame(type_descriptions), mentions


Training-only vocabulary: 17 categories, 50,000 TF-IDF terms


## 7. Impression features and candidate matching

`history_length` counts the supplied prior IDs, including repeats. `num_candidates` counts the full original slate before sampling. The two are stored once per event.

Time is the impression context, not article publication time or a permanent user trait. Keep the recorded timezone-naive values. With

$$
h=\text{hour}+\frac{\text{minute}}{60}+\frac{\text{second}}{3600},\qquad
\text{hour\_sin}=\sin\left(\frac{2\pi h}{24}\right),\quad
\text{hour\_cos}=\cos\left(\frac{2\pi h}{24}\right),
$$

near-midnight times remain close on the clock. Both coordinates are needed. Weekday/weekend values are exported but excluded from the default model-column list because training has no repeated weekly cycle.

Candidate-dependent time crosses are necessary for time to change the order within an impression. Preserve ZP's time × category and time × category-interest match terms. Category shares normalise over known training-vocabulary categories; subcategory shares use raw known parent/subcategory pairs. Unknowns never count as matching interests.

Entity similarities compare each candidate with its **supplied prior history**. Global variants average usable article vectors before cosine scoring. Type-specific history profiles average SY's unit article vectors; exact overlap is the share of candidate entities appearing anywhere in history. Raw absence and missing embeddings have separate flags. Scores are zero when a comparison cannot be made. Empty histories remain in every output.


In [ ]:
ALL_TIME_COLUMNS = ['hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'is_weekend']
SELECTED_TIME_COLUMNS = ['hour_sin', 'hour_cos'] + (
    ['weekday_sin', 'weekday_cos', 'is_weekend'] if USE_WEEKDAY_FEATURES else [])


def build_history_matrix(events, news_ids):
    index = pd.Index(news_ids)
    lengths = events['history_ids'].str.len().to_numpy(dtype='int64')
    flat = [n for history in events['history_ids'] for n in history]
    positions = index.get_indexer(flat)
    require((positions >= 0).all(), 'A supplied history article is missing from its partition news table')
    return sparse.csr_matrix((np.ones(len(flat), dtype='float32'),
        (np.repeat(np.arange(len(events), dtype='int64'), lengths), positions)),
        shape=(len(events), len(news_ids)))


def derive_event_features(events, num_candidates, news, encoded, history, incidence, type_vectors):
    out = events[['event_key', 'user_id', 'timestamp']].copy()
    out['history_length'] = events['history_length'].astype('int32')
    out['num_candidates'] = num_candidates.to_numpy(dtype='int32')  # Full original slate, never 5 sampled slots.
    stamp = events['timestamp']
    h = stamp.dt.hour + stamp.dt.minute / 60.0 + stamp.dt.second / 3600.0
    d = stamp.dt.dayofweek
    out['hour_of_day'] = stamp.dt.hour.astype('int8')
    out['fractional_hour'] = h.astype('float32')
    out['day_of_week'] = d.astype('int8')
    out['is_weekend'] = d.ge(5).astype('int8')
    out['hour_sin'] = np.sin(2 * np.pi * h / 24).astype('float32')
    out['hour_cos'] = np.cos(2 * np.pi * h / 24).astype('float32')
    out['weekday_sin'] = np.sin(2 * np.pi * d / 7).astype('float32')
    out['weekday_cos'] = np.cos(2 * np.pi * d / 7).astype('float32')
    cat_codes = encoded['category_id'].to_numpy(dtype='int32')
    onehot = np.eye(CATEGORY_COUNT, dtype='float32')[cat_codes]
    onehot[:, 0] = 0  # Unknown categories never become shared interests.
    category_counts = np.asarray(history @ onehot, dtype='float32')
    known_count = category_counts.sum(axis=1)
    shares = np.divide(category_counts, known_count[:, None],
                       out=np.zeros_like(category_counts), where=known_count[:, None] > 0)
    out['history_known_category_count'] = known_count.astype('int32')
    out['history_category_missing'] = (known_count == 0).astype('int8')
    for i, c in enumerate(HISTORY_CATEGORY_SHARE_COLUMNS):
        out[c] = shares[:, i]

    raw_subkeys = subcategory_key(news)
    sub_vocab = pd.Index(sorted(set(raw_subkeys) - {''}))
    sub_codes = sub_vocab.get_indexer(raw_subkeys)
    valid = sub_codes >= 0
    sub_inc = sparse.csr_matrix((np.ones(int(valid.sum()), dtype='float32'),
                                (np.flatnonzero(valid), sub_codes[valid])),
                               shape=(len(news), len(sub_vocab)))
    sub_profile = (history @ sub_inc).tocsr()
    sub_total = np.asarray(sub_profile.sum(axis=1)).ravel()
    out['history_subcategory_missing'] = (sub_total == 0).astype('int8')

    type_profiles, type_seen = {}, {}
    for name in ENTITY_TYPES:
        counts = (history @ incidence[name]).tocsr()
        out[f'history_{name}_missing'] = (np.asarray(counts.sum(axis=1)).ravel() == 0).astype('int8')
        type_seen[name] = counts.astype(bool).astype('float32')
        type_profiles[name] = normalise_rows(np.asarray(history @ type_vectors[name], dtype='float32'))
        out[f'history_{name}_embedding_missing'] = (np.linalg.norm(type_profiles[name], axis=1) == 0).astype('int8')
    return out, {'category_shares': shares, 'sub_codes': sub_codes, 'sub_profile': sub_profile,
                 'sub_total': sub_total, 'type_profiles': type_profiles, 'type_seen': type_seen}


def entity_history_profiles(history, vectors):
    result = {}
    for name, matrix in vectors.items():
        # Ethan: average only usable history vectors; repeated supplied items count again.
        valid = (np.linalg.norm(matrix, axis=1) > 0).astype('float32')
        count = np.asarray(history @ valid).ravel()
        total = np.asarray(history @ matrix, dtype='float32')
        mean = np.divide(total, count[:, None], out=np.zeros_like(total), where=count[:, None] > 0)
        result[name] = normalise_rows(mean)
    return result


def candidate_matching_features(batch, cache):
    # Labels and sampling metadata are not read. Repeated sampled rows are supported.
    keys = batch[KEYS].reset_index(drop=True)
    e = cache['event_index'].get_indexer(keys['event_key'])
    a = cache['news_index'].get_indexer(keys['news_id'])
    require((e >= 0).all() and (a >= 0).all(), 'Candidate references unknown event/article IDs')
    cat = cache['encoded']['category_id'].to_numpy(dtype='int32')[a]
    values = {}
    shares = cache['profiles']['category_shares'][e, cat]
    values['candidate_history_category_share'] = shares.astype('float32')
    # Base hour columns live once in event_features; one-hot category lives once in news_features.
    for t in SELECTED_TIME_COLUMNS:
        tvalue = cache['event_features'][t].to_numpy(dtype='float32')[e]
        for i, name in enumerate(CATEGORY_ONEHOT_COLUMNS):
            values[f'{t}__x__{name}'] = (tvalue * (cat == i)).astype('float32')
        values[f'{t}__x__history_category_share'] = (tvalue * shares).astype('float32')

    sub_codes = cache['profiles']['sub_codes'][a]
    known_sub = sub_codes >= 0
    sub_count = np.zeros(len(batch), dtype='float32')
    if known_sub.any():
        sub_count[known_sub] = np.asarray(cache['profiles']['sub_profile'][e[known_sub], sub_codes[known_sub]]).ravel()
    sub_total = cache['profiles']['sub_total'][e]
    values['candidate_history_subcategory_count'] = sub_count.astype('int32')
    values['candidate_history_subcategory_share'] = np.divide(
        sub_count, sub_total, out=np.zeros_like(sub_count), where=sub_total > 0).astype('float32')

    for name, matrix in cache['vectors'].items():
        candidate = cache['unit_vectors'][name][a]
        profile = cache['entity_history'][name][e]
        values[f'entity_similarity_{name}'] = np.clip(np.einsum('ij,ij->i', candidate, profile), -1, 1).astype('float32')

    for name in ENTITY_TYPES:
        candidate_inc = cache['incidence'][name][a]
        count = np.asarray(candidate_inc.sum(axis=1)).ravel()
        matches = np.asarray(cache['profiles']['type_seen'][name][e].multiply(candidate_inc).sum(axis=1)).ravel()
        values[f'{name}_entity_overlap'] = np.divide(matches, count, out=np.zeros_like(matches), where=count > 0).astype('float32')
        values[f'{name}_embedding_similarity'] = np.clip(np.einsum('ij,ij->i',
            cache['profiles']['type_profiles'][name][e], cache['type_vectors'][name][a]), -1, 1).astype('float32')
    out = pd.concat([keys, pd.DataFrame(values)], axis=1)
    require(np.isfinite(out.drop(columns=KEYS).to_numpy()).all(), 'Non-finite matching features')
    return out


## 8. Historical popularity/CTR with an explicit feedback protocol

All 21 article, subcategory, user and user–article features are retained, including 1/6/24-hour windows and smoothing masses 100 (subcategory) and 20 (article/user). Query times are strict `< t`: impressions at the same timestamp cannot see one another.

| Mode | Training | Validation | Test |
|---|---|---|---|
| **`train_only` (default)** | Earlier training events | Frozen training interactions | The same frozen training interactions |
| `frozen` (source Chris) | Earlier training events | Training interactions | Training + validation interactions before test starts |
| `online` (source Chris) | Earlier training events | Earlier events, including validation | Earlier events, including test feedback |

The latter two require `ALLOW_HELDOUT_FEEDBACK=True` and an agreed evaluation protocol. The default does not read validation/test label columns for feature construction. Enabling diagnostics reads validation labels only to evaluate features, not to fit them.

**Prior correction:** the source global CTR prior was computed from all training labels. Here the default prior is itself an expanding rate from strictly past **permitted** interactions; before any feedback it uses the fixed configured 0.04. This avoids a small current/future-training-label path through the smoothing constant. It is a documented construction change, so numeric CTR values during training will differ from the source notebook. Counts and per-partition query rules retain their meanings.

Statistical features always use complete candidate logs before negative sampling. A newly shown article starts from its topic/global smoothed rate, not a fabricated click probability.


In [ ]:
"""Article popularity and click-through features from interactions strictly before each impression.

Two modes share one implementation:
- online: every logged exposure and click with time < t, from any partition.
- frozen: training rows use time < t; rows in later partitions only see
  interactions before their partition starts (the interim-report rule).
The impression being scored never contributes, because it has time t.
"""
import numpy as np
import pandas as pd


WINDOW_HOURS = (1, 6, 24)
SMOOTHING = {"subcategory": 100.0, "article": 20.0, "user": 20.0}

ARTICLE_COLUMNS = [
    "art_prior_exposures", "art_prior_clicks", "art_prior_ctr",
    *[f"art_{kind}_{h}h" for h in WINDOW_HOURS for kind in ("exposures", "clicks")],
    "art_ctr_24h", "art_hours_since_first_exposure", "art_seen_before",
]
SUBCATEGORY_COLUMNS = ["subcat_prior_ctr"]
USER_COLUMNS = ["user_prior_events", "user_prior_exposures", "user_prior_clicks", "user_prior_ctr"]
USER_ARTICLE_COLUMNS = ["user_article_prior_exposures", "user_article_prior_clicks", "user_article_skipped_before"]
FEATURE_COLUMNS = ARTICLE_COLUMNS + SUBCATEGORY_COLUMNS + USER_COLUMNS + USER_ARTICLE_COLUMNS
FLAG_COLUMNS = ("art_seen_before", "user_article_skipped_before")
FLOAT_COLUMNS = ("art_prior_ctr", "art_ctr_24h", "art_hours_since_first_exposure", "subcat_prior_ctr", "user_prior_ctr")


def _distinct(values):
    """Distinct string keys plus each row's index into them (fast for categoricals)."""
    series = pd.Series(values, copy=False)
    if isinstance(series.dtype, pd.CategoricalDtype):
        codes = series.cat.codes.to_numpy()
        require((codes >= 0).all(), "Missing key in a categorical column")
        return series.cat.categories.astype(str).to_numpy(), codes
    codes, uniques = pd.factorize(series.astype(str))
    return np.asarray(uniques, dtype=object), codes


def _codes(values, table):
    keys, rows = _distinct(values)
    for key in keys:
        if key not in table:
            table[key] = len(table)
    return pd.Series(table).reindex(keys).to_numpy().astype("int64")[rows]


def build_exposure_log(partitions, subcategory_keys=None):
    """Compact integer log with one row per candidate exposure, partitions in time order.

    partitions maps name -> (events, candidates); rows keep each partition's
    candidate order, so features can be re-attached to its candidates table.
    subcategory_keys optionally maps name -> Series(news_id -> raw subcategory key).
    Unknown subcategories use the key "" and fall back to the global rate.
    """
    tables = {"article": {}, "user": {}, "subcat": {"": 0}}
    frames, previous_end, event_offset, origin = [], None, 0, None
    for index, (name, (events, candidates)) in enumerate(partitions.items()):
        require(events["event_key"].is_unique, f"Repeated event_key in {name}")
        event_pos = pd.Series(np.arange(len(events)), index=events["event_key"].astype(str).to_numpy())
        keys, rows = _distinct(candidates["event_key"])
        positions = event_pos.reindex(keys).to_numpy()
        require(not np.isnan(positions).any(), f"Candidates in {name} reference unknown events")
        event_of_row = positions.astype("int64")[rows]
        stamps = pd.to_datetime(events["timestamp"]).to_numpy()
        start, end = stamps.min(), stamps.max()
        require(previous_end is None or previous_end < start, f"{name} overlaps an earlier partition")
        previous_end = end
        origin = start if origin is None else origin
        users = _codes(events["user_id"], tables["user"])
        frame = pd.DataFrame({
            "partition": np.full(len(candidates), index, dtype="int8"),
            "event": event_of_row + event_offset,
            "article": _codes(candidates["news_id"], tables["article"]),
            "user": users[event_of_row],
            "label": candidates["label"].to_numpy(dtype="int64"),
            "timestamp": stamps[event_of_row],
        })
        if subcategory_keys is not None:
            keys, rows = _distinct(candidates["news_id"])
            sub = pd.Series(keys).map(subcategory_keys[name]).fillna("")
            frame["subcat"] = _codes(sub, tables["subcat"])[rows]
        else:
            frame["subcat"] = 0
        frames.append(frame)
        event_offset += len(events)
    log = pd.concat(frames, ignore_index=True)
    log["t"] = ((log.pop("timestamp") - origin) // pd.Timedelta(seconds=1)).astype("int64")
    log.attrs["partitions"] = list(partitions)
    return log


class _GroupStream:
    """Sorted (group, time) keys with cumulative clicks for O(log n) prior counts."""

    def __init__(self, group, t, clicks, span):
        self.span = span
        key = group * span + t
        order = np.argsort(key, kind="stable")
        self.keys = key[order]
        self.cum = np.concatenate([[0], np.cumsum(clicks[order])])

    def counts(self, group, upper, lower=None):
        base = group * self.span
        hi = np.searchsorted(self.keys, base + upper, "left")
        floor = base if lower is None else base + np.maximum(lower, 0)
        lo = np.minimum(np.searchsorted(self.keys, floor, "left"), hi)
        return hi - lo, self.cum[hi] - self.cum[lo]

    def first_time(self, group, upper):
        base = group * self.span
        start = np.searchsorted(self.keys, base, "left")
        hi = np.searchsorted(self.keys, base + upper, "left")
        seen = hi > start
        first = np.where(seen, self.keys[np.minimum(start, len(self.keys) - 1)] - base, -1)
        return seen, first




In [ ]:
def history_query_policy(log, mode):
    require(mode in ('train_only', 'frozen', 'online'), 'Unknown historical feature mode')
    t = log['t'].to_numpy(dtype='int64')
    train = log['partition'].to_numpy() == 0
    if mode == 'train_only':
        # Validation/test use only the same frozen training interactions.
        upper = np.where(train, t, int(t[train].max()) + 1)
        source = train
    elif mode == 'frozen':
        # Original Chris convention: test can use validation feedback.
        starts = log.groupby('partition', sort=False)['t'].transform('min').to_numpy()
        upper = np.where(train, t, np.minimum(t, starts))
        source = np.ones(len(log), dtype=bool)
    else:
        # Original Chris online convention: earlier feedback in any split.
        upper = t.copy()
        source = np.ones(len(log), dtype=bool)
    return upper, source


def compute_history_features(log, mode='train_only', prior_ctr=None):
    """Strictly-past popularity, windowed CTR, user and user-article counts.

    Default global prior is itself queried from strictly past permitted rows;
    GLOBAL_COLD_START_CTR is used when none exist. An explicit scalar prior_ctr
    is provided only for reproducibility checks against the source helper.
    """
    q, source = history_query_policy(log, mode)
    t = log['t'].to_numpy(dtype='int64')
    clicks = log['label'].to_numpy(dtype='int64')
    span = int(t.max()) + 2
    require(source.any(), 'No source interactions for historical features')
    make_stream = lambda group: _GroupStream(group[source], t[source], clicks[source], span)
    if prior_ctr is None:
        global_stream = _GroupStream(np.zeros(int(source.sum()), dtype='int64'),
                                    t[source], clicks[source], span)
        past_exposures, past_clicks = global_stream.counts(np.zeros(len(log), dtype='int64'), q)
        p0 = np.divide(past_clicks, past_exposures,
                       out=np.full(len(log), GLOBAL_COLD_START_CTR, dtype='float64'),
                       where=past_exposures > 0)
        del global_stream, past_exposures, past_clicks
    else:
        p0 = np.full(len(log), float(prior_ctr))
    sub = log['subcat'].to_numpy(dtype='int64')
    exposures, hits = make_stream(sub).counts(sub, q)
    m = SMOOTHING['subcategory']
    sub_rate = np.where(sub == 0, p0, (hits + m * p0) / (exposures + m))
    art = log['article'].to_numpy(dtype='int64')
    stream = make_stream(art)
    exposures, hits = stream.counts(art, q)
    m = SMOOTHING['article']
    out = {'art_prior_exposures': exposures, 'art_prior_clicks': hits,
           'art_prior_ctr': (hits + m * sub_rate) / (exposures + m)}
    for h in WINDOW_HOURS:
        out[f'art_exposures_{h}h'], out[f'art_clicks_{h}h'] = stream.counts(art, q, t - h * 3600)
    out['art_ctr_24h'] = (out['art_clicks_24h'] + m * sub_rate) / (out['art_exposures_24h'] + m)
    seen, first = stream.first_time(art, q)
    out['art_hours_since_first_exposure'] = np.where(seen, (t - first) / 3600, 0)
    out['art_seen_before'] = seen
    out['subcat_prior_ctr'] = sub_rate
    del stream
    user = log['user'].to_numpy(dtype='int64')
    stream = make_stream(user)
    exposures, hits = stream.counts(user, q)
    m = SMOOTHING['user']
    out['user_prior_exposures'], out['user_prior_clicks'] = exposures, hits
    out['user_prior_ctr'] = (hits + m * p0) / (exposures + m)
    first_row = ~pd.Series(log['event'].to_numpy()).duplicated().to_numpy()
    event_source = first_row & source
    event_stream = _GroupStream(user[event_source], t[event_source],
                                np.zeros(int(event_source.sum()), dtype='int64'), span)
    out['user_prior_events'], _ = event_stream.counts(user, q)
    del stream, event_stream
    pairs = pd.factorize(user * (int(art.max()) + 1) + art)[0].astype('int64')
    exposures, hits = make_stream(pairs).counts(pairs, q)
    out['user_article_prior_exposures'], out['user_article_prior_clicks'] = exposures, hits
    out['user_article_skipped_before'] = exposures > hits
    result = pd.DataFrame(index=log.index)
    for c in FEATURE_COLUMNS:
        dtype = 'int8' if c in FLAG_COLUMNS else 'float32' if c in FLOAT_COLUMNS else 'int32'
        result[c] = np.asarray(out[c]).astype(dtype)
    require(np.isfinite(result.to_numpy()).all(), 'Non-finite historical features')
    return result


## 9. Optional feature diagnostics

The following functions recompute training entity-type/lead-type CTR tables, validation univariate AUC/MIND-MRR/nDCG@5/10, confidence-vector agreement and paired weighting differences. The full 11-scheme prominence grid is controlled separately. Diagnostic results from the original notebooks are not copied as claims about this run.

The grid uses validation only and a fixed uniform-coverage subset, following the source comparison. A recommendation from that grid does not silently overwrite export parameters. Test labels are never used for diagnostic selection. These are feature screens, not trained-model results or proof of causal preference.


In [ ]:
"""Per-impression ranking metrics for quick feature checks on validation data.

Definitions follow the MIND evaluation script: AUC, MRR, nDCG@5 and nDCG@10
are computed inside each impression and then averaged over impressions. Ties
in AUC count one half; ties in the rank-based metrics are broken at random
with a fixed seed so equal scores never favour clicked candidates.
"""
import numpy as np
import pandas as pd


def impression_metrics(frame, score, group="event_key", label="label", seed=0, per_impression=False):
    """Mean metrics over impressions with at least one click and one non-click.

    With per_impression=True, return one row per impression instead, which
    supports paired comparisons between two scores on the same impressions.
    """
    scores = frame[score].to_numpy(dtype="float64")
    scores = np.where(np.isnan(scores), np.nanmin(scores) if np.isfinite(scores).any() else 0.0, scores)
    data = pd.DataFrame({"g": frame[group].to_numpy(), "y": frame[label].to_numpy(dtype="int64"), "s": scores})
    rng = np.random.default_rng(seed)
    data["tie"] = rng.random(len(data))
    data = data.sort_values(["g", "s", "tie"], ascending=[True, False, False], kind="stable")
    data["rank"] = data.groupby("g", sort=False).cumcount() + 1
    data["avg_rank"] = data.groupby("g", sort=False)["s"].rank(method="average", ascending=True)
    grouped = data.groupby("g", sort=False)
    n = grouped["y"].size()
    pos = grouped["y"].sum()
    valid = (pos > 0) & (pos < n)
    data["rr"] = data["y"] / data["rank"]
    data["gain"] = data["y"] / np.log2(data["rank"] + 1)
    data["pos_rank"] = data["y"] * data["avg_rank"]
    sums = data.groupby("g", sort=False).agg(rr=("rr", "sum"), pr=("pos_rank", "sum"))
    dcg = {k: data.loc[data["rank"].le(k)].groupby("g", sort=False)["gain"].sum().reindex(n.index, fill_value=0.0)
           for k in (5, 10)}
    discount = np.concatenate([[0.0], np.cumsum(1 / np.log2(np.arange(2, 12)))])
    auc = (sums["pr"] - pos * (pos + 1) / 2) / (pos * (n - pos))
    table = pd.DataFrame({"AUC": auc, "MRR": sums["rr"] / pos})
    for k in (5, 10):
        table[f"nDCG@{k}"] = dcg[k] / discount[np.minimum(pos.to_numpy(), k)]
    table = table.loc[valid]
    if per_impression:
        return table
    result = {"impressions": int(valid.sum()), **table.mean().to_dict()}
    return {key: (round(float(v), 4) if isinstance(v, float) else v) for key, v in result.items()}


def paired_difference(frame, score_a, score_b, metric="AUC", **kwargs):
    """Mean per-impression difference (b minus a) with a normal 95% interval."""
    a = impression_metrics(frame, score_a, per_impression=True, **kwargs)[metric]
    b = impression_metrics(frame, score_b, per_impression=True, **kwargs)[metric]
    diff = (b - a.reindex(b.index)).dropna()
    half = 1.96 * diff.std(ddof=1) / np.sqrt(len(diff))
    return {"metric": metric, "mean_difference": round(float(diff.mean()), 4),
            "ci_low": round(float(diff.mean() - half), 4), "ci_high": round(float(diff.mean() + half), 4),
            "impressions": int(len(diff))}


def univariate_table(frame, columns, sign=None, **kwargs):
    """Score candidates by one feature at a time; sign=-1 flips features expected to lower clicks."""
    sign = sign or {}
    rows = []
    for column in columns:
        scored = frame[["event_key", "label"]].assign(_s=frame[column].astype("float64") * sign.get(column, 1))
        rows.append({"feature": column, "direction": "lower is better" if sign.get(column, 1) < 0 else "higher is better",
                     **impression_metrics(scored, "_s", **kwargs)})
    return pd.DataFrame(rows)



In [ ]:
def read_saved_matching(partition):
    return pd.read_csv(STAGING / partition / 'candidate_features.csv.gz',
                       dtype={'event_key': 'string', 'news_id': 'string', 'candidate_position': 'int32'})


def run_partition_diagnostics(partition, cache):
    # Source notebook narrative results are not copied as new experimental evidence.
    if partition == 'test':
        return
    if partition == 'train':
        # Streaming pooled training checks, no full wide training table in memory.
        bins = Counter()
        lead = Counter()
        paths = STAGING / partition / 'candidate_features.csv.gz'
        matches = pd.read_csv(paths, chunksize=CANDIDATE_CHUNK_SIZE,
                              dtype={'event_key': 'string', 'news_id': 'string', 'candidate_position': 'int32'})
        lead_of = cache['news_features'].set_index('news_id')['title_start_type_group']
        for batch, matching in zip(candidate_chunks(partition, labels=True), matches, strict=True):
            require(batch[KEYS].reset_index(drop=True).equals(matching[KEYS].reset_index(drop=True)), 'Diagnostic alignment error')
            labels = batch['label'].to_numpy()
            for group in ('person', 'organisation', 'place', 'other', 'none'):
                selected = batch['news_id'].map(lead_of).eq(group).to_numpy()
                lead[(group, 'exposures')] += int(selected.sum())
                lead[(group, 'clicks')] += int(labels[selected].sum())
            for name in ENTITY_TYPES:
                a = cache['news_index'].get_indexer(batch['news_id'])
                e = cache['event_index'].get_indexer(batch['event_key'])
                missing = (cache['news_features'][f'{name}_entities_missing'].to_numpy()[a] > 0) | (
                    cache['event_features'][f'history_{name}_missing'].to_numpy()[e] > 0)
                overlap = matching[f'{name}_entity_overlap'].to_numpy()
                for group, selected in [('missing', missing), ('no_overlap', ~missing & (overlap == 0)),
                                         ('overlap', ~missing & (overlap > 0))]:
                    bins[(name, group, 'exposures')] += int(selected.sum())
                    bins[(name, group, 'clicks')] += int(labels[selected].sum())
        table = pd.DataFrame([{'type': name, 'group': group,
            'exposures': bins[name, group, 'exposures'], 'clicks': bins[name, group, 'clicks'],
            'ctr': bins[name, group, 'clicks'] / max(bins[name, group, 'exposures'], 1)}
            for name in ENTITY_TYPES for group in ('missing', 'no_overlap', 'overlap')])
        DIAGNOSTIC_TABLES['entity_types_train'] = table
        DIAGNOSTIC_TABLES['title_start_types_train'] = pd.DataFrame([
            {'group': group, 'exposures': lead[group, 'exposures'], 'clicks': lead[group, 'clicks'],
             'ctr': lead[group, 'clicks'] / max(lead[group, 'exposures'], 1)}
            for group in ('person', 'organisation', 'place', 'other', 'none')])
        return
    require(partition == 'validation', 'Diagnostics are restricted to train/validation')
    matching = read_saved_matching(partition)
    labels = read_candidates(partition, labels=True).astype({'event_key': 'string', 'news_id': 'string'})
    scored = merge_unique(matching, labels, KEYS)
    scores = [f'entity_similarity_{name}' for name in cache['vectors']]
    scores += ['candidate_history_category_share', 'candidate_history_subcategory_share']
    scores += [c for name in ENTITY_TYPES for c in (f'{name}_entity_overlap', f'{name}_embedding_similarity')]
    scored = merge_unique(scored, cache['news_features'][['news_id', 'has_entity_at_title_start',
        'title_start_type_group', 'title_entity_first_position', 'abstract_entity_first_position', 'title_entity_absent']], ['news_id'])
    extra = ['has_entity_at_title_start', 'title_entity_first_position', 'abstract_entity_first_position', 'title_entity_absent']
    if 'title_start_types_train' in DIAGNOSTIC_TABLES:
        group = DIAGNOSTIC_TABLES['title_start_types_train']
        p0 = group['clicks'].sum() / max(group['exposures'].sum(), 1)
        rate = pd.Series((group['clicks'] + 100 * p0) / (group['exposures'] + 100), index=group.index)
        rate.index = group['group']
        scored['lead_type_train_ctr'] = scored['title_start_type_group'].map(rate).fillna(p0)
        extra += ['lead_type_train_ctr']  # Diagnostic only, never a training target-encoded input.
    DIAGNOSTIC_TABLES['univariate_validation'] = univariate_table(scored, scores + extra,
        sign={'title_entity_first_position': -1, 'abstract_entity_first_position': -1, 'title_entity_absent': -1})
    multi = cache['news_features']['num_embedded_entities'].to_numpy() >= 2
    agreement = np.einsum('ij,ij->i', cache['unit_vectors']['uniform'][multi], cache['unit_vectors']['confidence'][multi])
    DIAGNOSTIC_TABLES['confidence_vector_agreement'] = pd.DataFrame([{
        'articles_with_multiple_embedded_entities': int(multi.sum()),
        'minimum_cosine': float(agreement.min()) if len(agreement) else np.nan,
        'median_cosine': float(np.median(agreement)) if len(agreement) else np.nan}])
    DIAGNOSTIC_TABLES['confidence_weighting_paired'] = pd.DataFrame([
        paired_difference(scored, 'entity_similarity_uniform', 'entity_similarity_confidence', metric)
        for metric in ('AUC', 'nDCG@10')])
    if RUN_PROMINENCE_GRID:
        run_prominence_grid(cache, labels)


def run_prominence_grid(cache, labels):
    schemes = {'uniform': (1.0, 0.0), 'abstract_half': (0.5, 0.0),
        'position_decay_1': (1.0, 1.0), 'abstract_half_decay_1': (0.5, 1.0),
        'abstract_half_decay_2': (0.5, 2.0), 'abstract_quarter_decay_2': (0.25, 2.0),
        'abstract_tenth_decay_1': (0.1, 1.0), 'abstract_tenth_decay_2': (0.1, 2.0),
        'title_only': (0.0, 0.0), 'title_only_decay_1': (0.0, 1.0), 'title_only_decay_2': (0.0, 2.0)}
    entity_index, matrix = embeddings_for('validation')
    e = cache['event_index'].get_indexer(labels['event_key'])
    a = cache['news_index'].get_indexer(labels['news_id'])
    scored = labels[['event_key', 'label']].copy()
    rows, covered = [], None
    for name, (abstract_weight, decay) in schemes.items():
        weights = mention_weights(cache['mentions'], abstract_weight=abstract_weight, decay=decay)
        vectors, missing = aggregate_entity_vectors(cache['news']['news_id'], cache['mentions'], weights, entity_index, matrix)
        profile = entity_history_profiles(cache['history'], {name: vectors})[name]
        unit = normalise_rows(vectors)
        scores = np.zeros(len(labels), dtype='float32')
        for start in range(0, len(labels), CANDIDATE_CHUNK_SIZE):
            part = slice(start, start + CANDIDATE_CHUNK_SIZE)
            scores[part] = np.einsum('ij,ij->i', profile[e[part]], unit[a[part]])
        scored[name] = scores
        valid = (np.linalg.norm(profile, axis=1)[e] > 0) & (np.linalg.norm(unit, axis=1)[a] > 0)
        if covered is None:
            covered = valid  # Fixed uniform-coverage subset, as in source Chris notebook.
        rows.append({'scheme': name, 'abstract_weight': abstract_weight, 'decay': decay,
            'article_coverage': float((missing == 0).mean()),
            **{f'all_{k}': v for k, v in impression_metrics(scored, name).items()},
            **{f'covered_{k}': v for k, v in impression_metrics(scored.loc[covered], name).items()}})
    table = pd.DataFrame(rows)
    DIAGNOSTIC_TABLES['prominence_grid_validation'] = table
    ranked = table.sort_values('covered_nDCG@10', ascending=False)
    best = ranked.iloc[0]['scheme']
    keep_abstract = ranked.loc[ranked['abstract_weight'].gt(0)].iloc[0]['scheme']
    DIAGNOSTIC_TABLES['prominence_grid_paired'] = pd.DataFrame([
        {'comparison': f'{name} minus uniform', **paired_difference(scored.loc[covered], 'uniform', name, metric)}
        for name in dict.fromkeys([best, keep_abstract, 'abstract_half_decay_1']) if name != 'uniform'
        for metric in ('AUC', 'nDCG@10')])
    print('Validation grid recommendations:', best, '(overall);', keep_abstract, '(keeps abstract).')
    print('Exported prominence parameters remain the explicit values in the configuration cell.')


def run_history_diagnostics(partition, mode, keys, features):
    labels = read_candidates(partition, labels=True)[['event_key', 'candidate_position', 'label']]
    scored = merge_unique(pd.concat([keys.reset_index(drop=True), features], axis=1),
                          labels, ['event_key', 'candidate_position'])
    sign = {'art_hours_since_first_exposure': -1, 'user_article_skipped_before': -1,
            'user_article_prior_exposures': -1}
    DIAGNOSTIC_TABLES[f'history_{mode}_validation'] = univariate_table(scored, list(FEATURE_COLUMNS), sign=sign)


## 10. Processing and verification

Partitions are processed one at a time; entity tables are cached by source. Candidate matching exports are chunked and keyed identically to the source slates. TF-IDF remains sparse. Vector files include a string `news_id` array and share the row order in `news_index.csv`.

Some candidate-level CSVs are large even when compressed. A whole-impression ranking evaluation must keep every candidate; chunk boundaries are I/O boundaries rather than evaluation groups. Group by `event_key` after collecting scores.


In [ ]:
import gzip

PARTITION_SUMMARY = []
TIME_COVERAGE = []
OFFSET_AUDIT = []
TEXT_SUMMARY = []
DIAGNOSTIC_TABLES = {}


def append_gzip_chunks(path, chunks):
    rows = 0
    with gzip.open(path, 'wt', encoding='utf-8', newline='', compresslevel=1) as stream:
        for frame in chunks:
            frame.to_csv(stream, index=False, header=rows == 0, float_format='%.8g')
            rows += len(frame)
    require(rows > 0, f'No candidate rows written to {path}')
    record_file(path, 'candidate', rows)
    return rows


def build_partition(partition):
    tick = time.time()
    folder = STAGING / partition
    print(f'Building {partition} ...', flush=True)
    news, events = read_news(partition), read_events(partition)
    counts = full_impression_counts(partition, events)
    if (PROCESSED_DIR / 'split_summary.csv').is_file():
        source_summary = pd.read_csv(PROCESSED_DIR / 'split_summary.csv').set_index('partition').loc[partition]
        require(len(events) == int(source_summary['events']), 'Event total differs from preprocessing summary')
        require(int(counts.sum()) == int(source_summary['candidate_exposures']), 'Candidate total differs from preprocessing summary')
    basic = derive_basic_article_features(news)
    encoded = encode_categories(news, category_vocab)
    entity_index, embedding_matrix = embeddings_for(partition)
    entity_features, vectors, incidence, type_vectors, descriptions, mentions = derive_entity_representations(
        news, entity_index, embedding_matrix)
    article = merge_unique(merge_unique(basic, encoded, ['news_id']), entity_features, ['news_id'])
    cat_codes = encoded['category_id'].to_numpy(dtype='int32')
    for i, c in enumerate(CATEGORY_ONEHOT_COLUMNS):
        article[c] = (cat_codes == i).astype('float32')
    history = build_history_matrix(events, news['news_id'])
    event_features, profiles = derive_event_features(events, counts, news, encoded, history, incidence, type_vectors)
    entity_history = entity_history_profiles(history, vectors)
    for name, matrix in entity_history.items():
        event_features[f'history_entity_vector_missing_{name}'] = (np.linalg.norm(matrix, axis=1) == 0).astype('int8')

    if partition == 'train':
        register_frame(article, 'article', 'ZP + SY + Chris + Ethan + Johannes', excluded=['news_id'])
        disabled_time = ['hour_of_day', 'fractional_hour', 'day_of_week'] + (
            ['weekday_sin', 'weekday_cos', 'is_weekend'] if not USE_WEEKDAY_FEATURES else [])
        register_frame(event_features, 'event', 'ZP + Johannes + SY + Chris',
            excluded=['event_key', 'user_id', 'timestamp'], disabled=disabled_time,
            descriptions={'num_candidates': 'Number of candidates in the complete original impression, before negative sampling',
                          'history_length': 'Length of the supplied prior clicked history, including repeated IDs'})
    require(article['news_id'].equals(news['news_id']), 'Article row order changed')
    write_frame(article, folder / 'news_features.csv', 'article')
    write_frame(event_features, folder / 'event_features.csv', 'event', date_format='%Y-%m-%d %H:%M:%S')
    write_frame(descriptions, folder / 'article_entity_ids.csv.gz', 'article metadata', compression='gzip')
    index = pd.DataFrame({'row': np.arange(len(news)), 'news_id': news['news_id']})
    write_frame(index, folder / 'news_index.csv', 'matrix row index')
    np.savez_compressed(folder / 'article_entity_vectors.npz', news_id=news['news_id'].to_numpy(dtype=str),
                        **vectors, **{f'type_{name}': values for name, values in type_vectors.items()})
    record_file(folder / 'article_entity_vectors.npz', 'article matrix', len(news))
    text_matrix = vectorizer.transform(title_abstract_text(news)).astype('float32').tocsr()
    require(text_matrix.shape[0] == len(news) and np.isfinite(text_matrix.data).all(), 'Invalid TF-IDF matrix')
    sparse.save_npz(folder / 'tfidf_title_abstract.npz', text_matrix)
    record_file(folder / 'tfidf_title_abstract.npz', 'article sparse matrix', len(news))
    TEXT_SUMMARY.append({'partition': partition, 'articles': len(news), 'vocabulary_size': text_matrix.shape[1],
                         'nonzero_values': text_matrix.nnz, 'zero_rows': int((np.diff(text_matrix.indptr) == 0).sum())})
    del text_matrix
    cache = dict(news=news, events=events, encoded=encoded, event_features=event_features,
                 news_features=article, news_index=pd.Index(news['news_id']), event_index=pd.Index(events['event_key']),
                 vectors=vectors, unit_vectors={k: normalise_rows(v) for k, v in vectors.items()},
                 entity_history=entity_history, incidence=incidence, type_vectors=type_vectors,
                 profiles=profiles, mentions=mentions, history=history)
    first = True

    def feature_chunks():
        nonlocal first
        for batch in candidate_chunks(partition):
            frame = candidate_matching_features(batch, cache)
            if partition == 'train' and first:
                register_frame(frame, 'candidate', 'ZP + SY + Chris + Ethan + Johannes', excluded=KEYS)
            first = False
            yield frame

    written = append_gzip_chunks(folder / 'candidate_features.csv.gz', feature_chunks())
    require(written == int(counts.sum()), f'{partition}: candidates were lost in matching export')
    if RUN_VALIDATION_DIAGNOSTICS:
        run_partition_diagnostics(partition, cache)
    dates = events['timestamp'].dt.normalize().drop_duplicates()
    TIME_COVERAGE.append({'partition': partition, 'distinct_dates': len(dates),
        'observed_weekdays': ', '.join(sorted(events['timestamp'].dt.day_name().unique())),
        'max_dates_per_weekday': int(dates.dt.dayofweek.value_counts().max()),
        'weekend_events': int(event_features['is_weekend'].sum())})
    for section in ('title', 'abstract'):
        total = matched = invalid = 0
        for text, annotations in zip(news[section], news[f'{section}_entities_parsed']):
            for annotation in annotations:
                for surface, offset in zip(annotation.get('SurfaceForms') or [], annotation.get('OccurrenceOffsets') or []):
                    total += 1
                    good = isinstance(offset, int) and 0 <= offset <= len(text) and isinstance(surface, str)
                    matched += int(good and text[offset:offset + len(surface)] == surface)
                    invalid += int(not good)
        OFFSET_AUDIT.append({'partition': partition, 'section': section, 'checked_occurrences': total,
                              'matching_surface_forms': matched, 'invalid_offset_pairs': invalid})
    PARTITION_SUMMARY.append({'partition': partition, 'events': len(events), 'candidates': written,
        'articles': len(news), 'earliest_timestamp': events['timestamp'].min(),
        'latest_timestamp': events['timestamp'].max(), 'metadata_missing_articles': int(news['metadata_missing'].sum()),
        'empty_histories': int(events['history_length'].eq(0).sum()),
        'articles_without_uniform_vector': int(article['entity_vector_missing_uniform'].sum()),
        'minutes': round((time.time() - tick) / 60, 2)})
    for filename in ('news.csv', 'events.csv', 'candidates.csv'):
        source_path = PROCESSED_DIR / partition / filename
        SOURCE_HASHES[str(source_path)] = sha256_file(source_path)
    print(f'{partition}: {len(news):,} articles, {len(events):,} events, {written:,} candidate rows', flush=True)
    del cache, news, events, history, incidence, type_vectors, entity_history, profiles
    gc.collect()


def build_historical_outputs():
    # Only read outcome columns in partitions whose feedback the selected protocol permits.
    label_parts = {'train'}
    if 'frozen' in HISTORY_MODES and 'test' in PARTITIONS:
        label_parts.add('validation')
    if 'online' in HISTORY_MODES:
        label_parts.update(PARTITIONS)
    inputs, subkeys, keys = {}, {}, {}
    for partition in PARTITIONS:
        events = read_events(partition)[['event_key', 'user_id', 'timestamp']]
        candidates = read_candidates(partition, labels=partition in label_parts)
        if 'label' not in candidates:
            candidates['label'] = np.int8(0)  # Excluded source rows; labels were not read from disk.
        news = read_news(partition)
        subkeys[partition] = pd.Series(subcategory_key(news).to_numpy(), index=news['news_id'].astype(str))
        inputs[partition] = events, candidates
        keys[partition] = candidates[KEYS].copy()
    log = build_exposure_log(inputs, subkeys)
    del inputs, subkeys, news, events, candidates
    gc.collect()
    offsets = {}
    offset = 0
    for partition in PARTITIONS:
        offsets[partition] = offset
        offset += len(keys[partition])
    for mode in HISTORY_MODES:
        print(f'Historical mode: {mode}', flush=True)
        frame = compute_history_features(log, mode=mode)
        if mode == HISTORY_MODES[0]:
            register_frame(frame, 'historical candidate', 'Chris', descriptions={
                c: 'Strictly past permitted interactions; ' + c.replace('_', ' ') for c in frame})
        for partition in PARTITIONS:
            start = offsets[partition]
            key_table = keys[partition]

            def chunks():
                for i in range(0, len(key_table), CANDIDATE_CHUNK_SIZE):
                    n = min(CANDIDATE_CHUNK_SIZE, len(key_table) - i)
                    yield pd.concat([key_table.iloc[i:i+n].reset_index(drop=True),
                                     frame.iloc[start+i:start+i+n].reset_index(drop=True)], axis=1)

            path = STAGING / partition / f'candidate_history_{mode}.csv.gz'
            written = append_gzip_chunks(path, chunks())
            require(written == len(key_table), 'Historical output lost rows')
            if RUN_VALIDATION_DIAGNOSTICS and partition == 'validation':
                run_history_diagnostics(partition, mode, key_table,
                                        frame.iloc[start:start+len(key_table)].reset_index(drop=True))
        del frame
        gc.collect()
    return {'modes': list(HISTORY_MODES), 'label_columns_read_from': sorted(label_parts),
            'global_prior': 'strictly past permitted exposures/clicks, fixed cold-start fallback',
            'cold_start_prior': GLOBAL_COLD_START_CTR, 'smoothing': SMOOTHING,
            'window_hours': list(WINDOW_HOURS)}


In [ ]:
def run_smoke_tests():
    # A slate with repeated history, an empty history, and missing metadata.
    events = pd.DataFrame({'event_key': pd.Series(['e1', 'e2'], dtype='string'),
        'user_id': ['U1', 'U2'], 'timestamp': pd.to_datetime(['2019-11-09 23:59:30', '2019-11-10 00:00:00']),
        'history_ids': [['N1', 'N1', 'N2'], []], 'history_length': [3, 0]})
    history = build_history_matrix(events, pd.Series(['N1', 'N2']))
    np.testing.assert_array_equal(history.toarray(), [[2, 1], [0, 0]])
    # Cold/unknown subcategories preserve their known parent category.
    training = pd.DataFrame({'news_id': ['N1', 'N2'], 'category': ['news', 'news'],
                              'subcategory': ['local', 'local'], 'metadata_missing': False})
    vocab = fit_category_vocab(training, min_articles=1)
    unseen = training.iloc[[0]].assign(news_id='N3', subcategory='new-topic')
    encoded = encode_categories(unseen, vocab)
    require(encoded['category_id'].iloc[0] > 0 and encoded['subcategory_id'].iloc[0] > 0,
            'Known category has no reserved rare-subcategory backoff')
    require(encoded['subcategory_backoff'].iloc[0] == 1, 'Missing subcategory backoff flag')
    # Same-second events must not leak into each other, including the global CTR prior.
    log = pd.DataFrame({'partition': [0, 0, 0, 1, 2], 'event': [0, 1, 2, 3, 4],
        'article': [0, 0, 0, 0, 0], 'user': [0, 0, 0, 0, 0], 'subcat': [1, 1, 1, 1, 1],
        't': [0, 0, 10, 20, 30], 'label': [1, 0, 0, 1, 1]})
    a = compute_history_features(log, mode='train_only')
    require(a['art_prior_exposures'].tolist() == [0, 0, 2, 3, 3], 'Past-count / train-only policy is incorrect')
    changed = log.copy()
    changed.loc[changed['t'].ge(10), 'label'] = 1 - changed.loc[changed['t'].ge(10), 'label']
    b = compute_history_features(changed, mode='train_only')
    pd.testing.assert_frame_equal(a.iloc[:3], b.iloc[:3])
    heldout = log.copy()
    heldout.loc[heldout['partition'].gt(0), 'label'] = 1 - heldout.loc[heldout['partition'].gt(0), 'label']
    pd.testing.assert_frame_equal(a, compute_history_features(heldout, mode='train_only'))
    require(np.isfinite(a.to_numpy()).all(), 'Historical features contain non-finite values')
    print('Smoke checks passed: repeated/empty histories, category backoff, same-second exclusion, future-label invariance and train-only held-out-label invariance.')


run_smoke_tests()


Smoke checks passed: repeated/empty histories, category backoff, same-second exclusion, future-label invariance and train-only held-out-label invariance.


In [ ]:
for partition in PARTITIONS:
    build_partition(partition)
HISTORY_METADATA = build_historical_outputs()


def verify_exported_bundle():
    for partition in PARTITIONS:
        folder = STAGING / partition
        news = pd.read_csv(folder / 'news_features.csv', dtype={'news_id': 'string'}, keep_default_na=False)
        events = pd.read_csv(folder / 'event_features.csv', dtype={'event_key': 'string'}, keep_default_na=False)
        require(news['news_id'].is_unique and events['event_key'].is_unique, 'Nonunique feature lookup keys')
        for name, frame in [('article', news), ('event', events)]:
            columns = [c for c, spec in FEATURE_REGISTRY.items() if spec['level'] == name and spec['numeric']]
            require(np.isfinite(frame[columns].to_numpy(dtype='float64')).all(), f'{partition}: non-finite {name} features')
        index = pd.read_csv(folder / 'news_index.csv', dtype={'news_id': 'string'})
        require(index['news_id'].equals(news['news_id']), 'Matrix/news index mismatch')
        matrix = sparse.load_npz(folder / 'tfidf_title_abstract.npz')
        require(matrix.shape == (len(news), len(vocabulary)), 'Wrong TF-IDF shape')
        with np.load(folder / 'article_entity_vectors.npz', allow_pickle=False) as arrays:
            require(np.array_equal(arrays['news_id'], news['news_id'].to_numpy(dtype=str)), 'Entity-vector index mismatch')
            for name in arrays.files:
                if name != 'news_id':
                    require(arrays[name].shape == (len(news), ENTITY_DIMENSION), f'Wrong {name} shape')
                    require(np.isfinite(arrays[name]).all(), f'Non-finite {name} values')
        expected_rows = next(row['candidates'] for row in PARTITION_SUMMARY if row['partition'] == partition)
        for filename in ['candidate_features.csv.gz', *[f'candidate_history_{m}.csv.gz' for m in HISTORY_MODES]]:
            written = 0
            chunks = pd.read_csv(folder / filename, chunksize=CANDIDATE_CHUNK_SIZE,
                                 dtype={'event_key': 'string', 'news_id': 'string', 'candidate_position': 'int32'})
            for original, feature in zip(candidate_chunks(partition), chunks, strict=True):
                require(original[KEYS].reset_index(drop=True).equals(feature[KEYS].reset_index(drop=True)),
                        f'{partition}/{filename}: row order or candidate identity changed')
                require('label' not in feature, 'Click target leaked into a feature output')
                require(not feature.columns.duplicated().any(), 'Duplicate output column names')
                require(np.isfinite(feature.drop(columns=KEYS).to_numpy(dtype='float64')).all(), 'Non-finite candidate features')
                written += len(feature)
            require(written == expected_rows, 'Candidate total differs from source')
    print('Export checks passed: keys, complete candidate lists, matrix alignment, finite values and separate targets.')


verify_exported_bundle()


Building train ...
train: 43,370 articles, 95,071 events, 3,389,342 candidate rows
Building validation ...
validation: 38,421 articles, 61,894 events, 2,454,102 candidate rows
Building test ...
test: 42,416 articles, 73,152 events, 2,740,998 candidate rows
Historical mode: train_only
Export checks passed: keys, complete candidate lists, matrix alignment, finite values and separate targets.


## 11. Publish one reproducible output bundle

| File | Level / purpose |
|---|---|
| `<partition>/news_features.csv` | One article row: counts, flags, category/subcategory encodings, entity confidence/types/prominence and category one-hots |
| `<partition>/event_features.csv` | One impression row: history counts, original slate size, time features, category profiles and history-missing flags |
| `<partition>/candidate_features.csv.gz` | Complete candidate rows: time crosses, history-category/subcategory matches, global and type-specific entity matching |
| `<partition>/candidate_history_<mode>.csv.gz` | Complete candidate rows: selected historical feedback-policy variant |
| `<partition>/news_index.csv` | Shared row index for all article matrices |
| `<partition>/article_entity_vectors.npz` | Uniform/confidence/prominence/title-only vectors and the three entity-type vectors |
| `<partition>/article_entity_ids.csv.gz` | Descriptive per-type exact entity IDs; not numeric predictors |
| `<partition>/tfidf_title_abstract.npz` | Sparse article text vectors |
| Root vocabularies/vectorizer, dictionary, manifest, summaries, aliases and merge notes | Fitted transforms, provenance, column semantics, integrity checks and overlap decisions |

The build is published only after output checks pass. If execution fails, the temporary `*-building-*` folder is an incomplete checkpoint, not the agreed group version. Keep the original input version unchanged. Pick a fresh output version for a later rebuild.


In [ ]:
# Matrix inputs are documented separately from scalar CSV model inputs.
matrix_definitions = [dict(feature='tfidf_title_abstract', level='article matrix',
    source_notebook='SY', dtype=f'sparse float32[{len(vocabulary)}]', numeric=False,
    default_model_input=False, description='Training-fitted title + abstract TF-IDF; rows follow news_index.csv')]
for name in [*VECTOR_SCHEMES, 'confidence', *[f'type_{t}' for t in ENTITY_TYPES]]:
    owner = 'Ethan + Johannes + Chris' if name == 'uniform' else 'Johannes' if name == 'confidence' else 'SY' if name.startswith('type_') else 'Chris'
    matrix_definitions.append(dict(feature=f'article_entity_vector_{name}', level='article matrix',
        source_notebook=owner, dtype=f'float32[{ENTITY_DIMENSION}]', numeric=False,
        default_model_input=False, description=f'Array {name} in article_entity_vectors.npz; rows follow news_index.csv'))
dictionary = pd.DataFrame([*FEATURE_REGISTRY.values(), *matrix_definitions])
file_of = {'article': '<partition>/news_features.csv', 'event': '<partition>/event_features.csv',
    'candidate': '<partition>/candidate_features.csv.gz',
    'historical candidate': '<partition>/candidate_history_<mode>.csv.gz'}
dictionary['file'] = dictionary['level'].map(file_of).fillna('<partition>/*.npz + news_index.csv')
write_frame(dictionary, STAGING / 'feature_dictionary.csv', 'documentation')
write_frame(pd.DataFrame(PARTITION_SUMMARY), STAGING / 'summary.csv', 'audit')
write_frame(pd.DataFrame(TIME_COVERAGE), STAGING / 'time_coverage.csv', 'audit')
write_frame(pd.DataFrame(OFFSET_AUDIT), STAGING / 'offset_audit.csv', 'audit')
write_frame(pd.DataFrame(TEXT_SUMMARY), STAGING / 'tfidf_summary.csv', 'audit')
for name, table in DIAGNOSTIC_TABLES.items():
    write_frame(table, STAGING / f'diagnostic_{name}.csv', 'optional diagnostic')
write_frame(pd.DataFrame(ALIAS_RECORDS), STAGING / 'feature_aliases.csv', 'documentation')
write_frame(pd.DataFrame(MERGE_DECISIONS), STAGING / 'merge_decisions.csv', 'documentation')
(STAGING / 'merge_notes.md').write_text(MERGE_NOTES, encoding='utf-8')
record_file(STAGING / 'merge_notes.md', 'documentation')

NUMERIC_MODEL_COLUMNS = [c for c, spec in FEATURE_REGISTRY.items() if spec['default_model_input']]
CATEGORICAL_MODEL_COLUMNS = ['category_id', 'subcategory_id']
require(len(NUMERIC_MODEL_COLUMNS) == len(set(NUMERIC_MODEL_COLUMNS)), 'Duplicate dense model inputs')
inventory = pd.DataFrame(FILE_INVENTORY)
inventory.to_csv(STAGING / 'file_inventory.csv', index=False)
provenance = {
    'pipeline': 'BT4222 unified feature engineering',
    'feature_version': OUTPUT_DIR.name,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_repository': 'JariusF10-8/bt4222', 'source_commit': SOURCE_COMMIT,
    'source_notebooks': SOURCE_NOTEBOOKS, 'source_helpers': SOURCE_HELPERS,
    'source_processed_directory': str(PROCESSED_DIR),
    'source_dataset_version': source_manifest.get('dataset_version'),
    'source_manifest_sha256': sha256_file(manifest_path),
    'source_file_sha256': SOURCE_HASHES,
    'partitions': PARTITIONS, 'complete_input': source_manifest.get('complete'),
    'negative_sampling': 'none during feature construction; use src.sampling at model training only',
    'timestamp_timezone': 'Unspecified; source-naive timestamps preserved without conversion',
    'weekday_model_inputs_enabled': USE_WEEKDAY_FEATURES,
    'time_model_columns': SELECTED_TIME_COLUMNS,
    'category_min_articles': MIN_SUBCATEGORY_ARTICLES,
    'category_unknown_id': 0, 'subcategory_rare_bucket_policy': 'one reserved bucket for every training category',
    'vocabularies_fitted_on': 'train/news.csv, excluding metadata_missing rows',
    'tfidf_fit_articles': FIT_ARTICLES,
    'tfidf_params': TFIDF_PARAMS, 'tfidf_vocabulary_size': len(vocabulary),
    'entity_sources': {p: EMBEDDING_SOURCES[p] for p in PARTITIONS},
    'embedding_dimension': ENTITY_DIMENSION, 'shared_entity_embedding_audit': EMBEDDING_AUDIT,
    'entity_variants': {'uniform': 'Distinct embedded Wikidata IDs; one unweighted mean',
        'confidence': 'Distinct embedded IDs; maximum annotation confidence per ID',
        **{k: {'abstract_weight': a, 'position_decay': d} for k, (a, d) in VECTOR_SCHEMES.items() if k != 'uniform'},
        'type_vectors': 'SY equal-weight unit article vectors, kept separate from ordinary entity means'},
    'history_features': HISTORY_METADATA,
    'numeric_model_columns': NUMERIC_MODEL_COLUMNS,
    'categorical_model_columns': CATEGORICAL_MODEL_COLUMNS,
    'excluded_from_model_inputs': ['event_key', 'candidate_position', 'news_id', 'user_id', 'timestamp', 'label', 'group_id', 'slot'],
    'optional_diagnostics': {'validation': RUN_VALIDATION_DIAGNOSTICS, 'prominence_grid': RUN_PROMINENCE_GRID,
                             'test_labels_used_for_feature_evaluation': False},
    'libraries': {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__,
                  'scipy': scipy.__version__, 'scikit_learn': sklearn.__version__},
    'files': FILE_INVENTORY,
    'validation': 'Input and output integrity checks completed; optional feature diagnostics are not trained model results',
}
(STAGING / 'manifest.json').write_text(json.dumps(provenance, indent=2, default=list) + '\n', encoding='utf-8')
require(not OUTPUT_DIR.exists(), 'Output version appeared during the build; choose a new version')
STAGING.rename(OUTPUT_DIR)
FEATURE_DIR = OUTPUT_DIR
display(pd.DataFrame(PARTITION_SUMMARY))
if TIME_COVERAGE[0]['max_dates_per_weekday'] < 2:
    print('Weekly context remains exploratory: training contains no repeated weekday cycle.')
print(f'Completed unified feature bundle: {FEATURE_DIR}')
print(f'{len(NUMERIC_MODEL_COLUMNS)} available default scalar inputs; category/subcategory IDs are separate categorical inputs.')
print(f'Elapsed minutes: {(time.time() - STARTED) / 60:.1f}')


,partition,events,candidates,articles,earliest_timestamp,latest_timestamp,metadata_missing_articles,empty_histories,articles_without_uniform_vector,minutes
0,train,95071,3389342,43370,2019-11-09 00:00:19,2019-11-12 23:59:25,0,1735,5999,5.99
1,validation,61894,2454102,38421,2019-11-13 00:01:00,2019-11-14 23:59:13,0,1503,5605,4.53
2,test,73152,2740998,42416,2019-11-15 00:00:01,2019-11-15 23:58:03,0,2214,5892,4.87


Weekly context remains exploratory: training contains no repeated weekday cycle.
Completed unified feature bundle: /content/drive/MyDrive/BT4222 Group Project/outputs/features/unified_v1
165 available default scalar inputs; category/subcategory IDs are separate categorical inputs.
Elapsed minutes: 22.1


## 12. Use the unified features for modelling

The helper below joins using `many_to_one` against unique right-hand lookups. This preserves replacement-sampled negatives. It keeps `label`, `group_id` and `slot` as metadata; none enters the exported model-column lists. Do not deduplicate the sampled left-hand table, or blindly pass every numeric column to a model.

`iter_model_batches('train')` is a memory-conscious reader of complete, aligned candidate features. Use `assemble_model_frame` with the output of `src.sampling.sample_negatives` for sampled training. In model code, keep `y = batch['label']`, select `NUMERIC_MODEL_COLUMNS` explicitly, and use `category_id`/`subcategory_id` as categorical embedding inputs. Fit any scalers or SVD on training only. TF-IDF stays sparse; article matrices use `news_index.csv`.

For an attention/LSTM history encoder, retain individual historical article vectors from the input `events.csv`; the pooled similarity features do not replace that sequence. Choose one global entity-vector variant for the shared article encoder and compare others in controlled ablations.


In [ ]:
# Current-impression outcome counts must not be model inputs.

# Excluded due to target leakage
EXCLUDED_OUTCOME_COUNTS = {'num_clicked', 'num_unclicked'}

NUMERIC_MODEL_COLUMNS = [
    column for column in NUMERIC_MODEL_COLUMNS
    if column not in EXCLUDED_OUTCOME_COUNTS
]

CATEGORICAL_MODEL_COLUMNS = [
    column for column in CATEGORICAL_MODEL_COLUMNS
    if column not in EXCLUDED_OUTCOME_COUNTS
]


def remove_outcome_counts(frame):
    """Remove current-impression outcome counts if present."""
    columns = [
        column for column in frame.columns
        if column in EXCLUDED_OUTCOME_COUNTS
    ]
    return frame.drop(columns=columns) if columns else frame


def assemble_model_frame(
    candidate_rows,
    article_features,
    event_features,
    matching_features,
    historical_features,
):
    """Merge on stable keys, preserving repeated negative samples.

    label/group_id/slot remain as target or grouping metadata.
    Current-impression outcome counts are removed before merging.
    """
    candidate_rows = remove_outcome_counts(
        candidate_rows.reset_index(drop=True)
    )
    article_features = remove_outcome_counts(article_features)
    event_features = remove_outcome_counts(event_features)
    matching_features = remove_outcome_counts(matching_features)
    historical_features = remove_outcome_counts(historical_features)

    result = merge_unique(candidate_rows, matching_features, KEYS)
    result = merge_unique(result, historical_features, KEYS)
    result = merge_unique(result, article_features, ['news_id'])
    result = merge_unique(result, event_features, ['event_key'])

    require(
        not EXCLUDED_OUTCOME_COUNTS.intersection(result.columns),
        'Current-impression outcome counts remain in the model frame',
    )
    require(
        result[
            NUMERIC_MODEL_COLUMNS + CATEGORICAL_MODEL_COLUMNS
        ].notna().all().all(),
        'A feature join did not cover every candidate',
    )
    require(
        result[KEYS].equals(candidate_rows[KEYS]),
        'Candidate order changed',
    )
    return result


def iter_model_batches(partition, history_mode=None, chunk_size=None):
    """Stream complete candidate lists with labels for modelling/evaluation.

    Evaluate test only after fixing the model.
    For sampled training, use assemble_model_frame with sampled rows
    and unique full feature lookups; retain repeated sampled rows.
    """
    mode = history_mode or HISTORY_MODES[0]
    require(mode in HISTORY_MODES, f'History mode {mode} was not exported')

    size = chunk_size or CANDIDATE_CHUNK_SIZE
    folder = FEATURE_DIR / partition

    news = pd.read_csv(
        folder / 'news_features.csv',
        dtype={'news_id': 'string'},
        keep_default_na=False,
    )
    events = pd.read_csv(
        folder / 'event_features.csv',
        dtype={'event_key': 'string', 'user_id': 'string'},
        keep_default_na=False,
    )

    types = {
        'event_key': 'string',
        'news_id': 'string',
        'candidate_position': 'int32',
    }
    matching = pd.read_csv(
        folder / 'candidate_features.csv.gz',
        dtype=types,
        chunksize=size,
    )
    history = pd.read_csv(
        folder / f'candidate_history_{mode}.csv.gz',
        dtype=types,
        chunksize=size,
    )
    source = candidate_chunks(
        partition,
        labels=True,
        chunk_size=size,
    )

    for candidates, match, prior in zip(
        source, matching, history, strict=True
    ):
        candidate_keys = candidates[KEYS].reset_index(drop=True)

        require(
            candidate_keys.equals(match[KEYS].reset_index(drop=True)),
            'Matching-file alignment error',
        )
        require(
            candidate_keys.equals(prior[KEYS].reset_index(drop=True)),
            'History-file alignment error',
        )

        yield assemble_model_frame(
            candidates, news, events, match, prior
        )


# Small TRAIN preview; no test labels are read here.
preview = next(iter_model_batches('train', chunk_size=8))

display(preview[[
    'event_key',
    'news_id',
    'title_length',
    'abstract_length',
    'num_candidates',
    'hour_sin',
    'candidate_history_category_share',
    'entity_similarity_uniform',
]])

print('num_clicked and num_unclicked are excluded from model inputs.')
print('Keep label as the target; select model inputs explicitly.')

,event_key,news_id,title_length,abstract_length,num_candidates,hour_sin,candidate_history_category_share,entity_similarity_uniform
0,MINDsmall_train/behaviors.tsv:20112,N54300,8,8,31,0.001382,0.000000,0.000000
1,MINDsmall_train/behaviors.tsv:20112,N46057,7,14,31,0.001382,0.400000,0.381341
2,MINDsmall_train/behaviors.tsv:20112,N57005,12,17,31,0.001382,0.400000,0.000000
3,MINDsmall_train/behaviors.tsv:20112,N52154,9,53,31,0.001382,0.000000,0.000000
4,MINDsmall_train/behaviors.tsv:20112,N57099,7,42,31,0.001382,0.400000,0.539232
5,MINDsmall_train/behaviors.tsv:20112,N31002,7,25,31,0.001382,0.466667,0.436255
6,MINDsmall_train/behaviors.tsv:20112,N37108,9,58,31,0.001382,0.400000,0.768739
7,MINDsmall_train/behaviors.tsv:20112,N58075,9,24,31,0.001382,0.400000,0.653058


num_clicked and num_unclicked are excluded from model inputs.
Keep label as the target; select model inputs explicitly.


Example for a small batch (the reader deliberately loads labels only at the modelling stage):

```python
for batch in iter_model_batches('train', history_mode='train_only'):
    X_numeric = batch[NUMERIC_MODEL_COLUMNS].to_numpy(dtype='float32')
    X_category = batch[CATEGORICAL_MODEL_COLUMNS].to_numpy(dtype='int64')
    y = batch['label'].to_numpy(dtype='float32')
    # Train your selected model here.
```

For sampled training, first draw from the **full** training candidate table with `seed=42+epoch`, then attach features using unique full lookups. Keep groups/slots if using a listwise objective. BCE remains compatible with attention. Validation/test ranking always uses complete original slates. The supplied iterator is a loader, not a sampling or batching policy for model optimisation.

The source notebooks' large diagnostic outputs and local-machine paths are intentionally not copied. This notebook contains one executable implementation and generated audit documents; it does not change the five originals or publish anything to GitHub.


## Source snapshot

Commit: `7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f` (main, inspected 30 October 2026).

- [feature_engine_sy.ipynb](https://github.com/JariusF10-8/bt4222/blob/7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f/notebooks/features/feature_engine_sy.ipynb) — blob `97f17f4ffec1067acd61eb1ca51ae70b01564062`
- [feature_engineering_chris.ipynb](https://github.com/JariusF10-8/bt4222/blob/7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f/notebooks/features/feature_engineering_chris.ipynb) — blob `71cd5070ed23c14233432cae35cf4001bf41988f`
- [feature_engineering_ethan.ipynb](https://github.com/JariusF10-8/bt4222/blob/7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f/notebooks/features/feature_engineering_ethan.ipynb) — blob `492ed72204979dc19121260e5412b9d42ca0c73d`
- [feature_engineering_johannes.ipynb](https://github.com/JariusF10-8/bt4222/blob/7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f/notebooks/features/feature_engineering_johannes.ipynb) — blob `cfee6815bc1c85ea020760f6b3a1775f2a91a3b7`
- [feature_engineering_zp.ipynb](https://github.com/JariusF10-8/bt4222/blob/7a64cbf1e174eb1a01fcb79cb9b787b6058aa86f/notebooks/features/feature_engineering_zp.ipynb) — blob `7244dd45e64ef2270bb3b1e3ef1464b54717ab69`

The manifest also records the supporting source-helper blob IDs, runtime input/embedding checksums, fitted parameters and active feedback policy. Validation of this deliverable uses synthetic fixtures generated by the canonical preprocessing helper, plus differential checks against original feature functions. Run on the shared full dataset for actual group outputs; no full-data execution result is claimed here.
